Environment Audit.

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0 — ENVIRONMENT AUDIT + TARGETED INSTALLER
#
# Checks all packages required for the full 5-cell pipeline.
# Prints a targeted pip install command based on what is missing.
# Does NOT install anything automatically — you copy and run the output.
# ─────────────────────────────────────────────────────────────────────────────

import importlib
import sys
import subprocess
from datetime import datetime, timezone

print("=" * 70)
print("  CELL 0 — ENVIRONMENT AUDIT")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"  Python: {sys.version}")
print("=" * 70)

# ─────────────────────────────────────────────────────────────────────────────
# PACKAGES TO CHECK
# import_name       : what you import in Python
# install_name      : what you pass to pip install
# version_attr      : attribute to read version from (optional)
# ─────────────────────────────────────────────────────────────────────────────

REQUIRED = [
    # Core
    {"import": "boto3",          "pip": "boto3",                   "note": "S3 client"},
    {"import": "cv2",            "pip": "opencv-python-headless",  "note": "Frame extraction"},
    {"import": "numpy",          "pip": "numpy",                   "note": "Array ops"},
    {"import": "PIL",            "pip": "Pillow",                  "note": "Image processing"},
    {"import": "tqdm",           "pip": "tqdm",                    "note": "Progress bars"},

    # RetinaFace pipeline
    {"import": "retinaface",     "pip": "retina-face",             "note": "Face detection"},
    {"import": "imagehash",      "pip": "imagehash",               "note": "pHash dedup"},
    {"import": "pybktree",       "pip": "pybktree",                "note": "BK tree dedup"},

    # Identity graph
    {"import": "scipy",          "pip": "scipy",                   "note": "Connected components fallback"},

    # Utilities
    {"import": "requests",       "pip": "requests",                "note": "HTTP downloads"},
    {"import": "sagemaker",      "pip": "sagemaker",               "note": "SageMaker session"},
]

# ─────────────────────────────────────────────────────────────────────────────
# SYSTEM TOOLS CHECK
# ─────────────────────────────────────────────────────────────────────────────

SYSTEM_TOOLS = ["unzip", "wget"]

print(f"\n  {'Package':<30} {'Status':<12} {'Note'}")
print(f"  {'-'*30} {'-'*12} {'-'*30}")

missing_pip   = []
available     = []

for pkg in REQUIRED:
    try:
        mod = importlib.import_module(pkg["import"])
        version = getattr(mod, "__version__", "installed")
        print(f"  {pkg['pip']:<30} {'OK':>10}   {version:<15} {pkg['note']}")
        available.append(pkg["pip"])
    except ImportError:
        print(f"  {pkg['pip']:<30} {'MISSING':>10}            {pkg['note']}")
        missing_pip.append(pkg["pip"])

print(f"\n  {'System Tool':<30} {'Status':<12}")
print(f"  {'-'*30} {'-'*12}")

missing_tools = []
for tool in SYSTEM_TOOLS:
    result = subprocess.run(["which", tool], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  {tool:<30} {'OK':>10}   {result.stdout.strip()}")
    else:
        print(f"  {tool:<30} {'MISSING':>10}")
        missing_tools.append(tool)

# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'=' * 70}")
print(f"  AUDIT SUMMARY")
print(f"{'=' * 70}")
print(f"  Packages available : {len(available)}")
print(f"  Packages missing   : {len(missing_pip)}")
print(f"  System tools OK    : {len(SYSTEM_TOOLS) - len(missing_tools)}/{len(SYSTEM_TOOLS)}")

if not missing_pip and not missing_tools:
    print(f"\n  ALL CLEAR — environment is ready. Safe to run Cell 1.")
else:
    print(f"\n  ACTION REQUIRED — copy and run the install command below:\n")

    if missing_pip:
        pip_cmd = f"!pip install -q {' '.join(missing_pip)}"
        print(f"  Python packages:")
        print(f"  {pip_cmd}")

    if missing_tools:
        tool_cmd = f"!apt-get install -y {' '.join(missing_tools)}"
        print(f"\n  System tools:")
        print(f"  {tool_cmd}")

    print(f"\n  After installing, restart the kernel and rerun this cell to confirm.")

print(f"{'=' * 70}")

  CELL 0 — ENVIRONMENT AUDIT
  2026-05-09 08:40:08 UTC
  Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:42:22) [GCC 14.3.0]

  Package                        Status       Note
  ------------------------------ ------------ ------------------------------
  boto3                                  OK   1.43.4          S3 client
  opencv-python-headless                 OK   4.11.0          Frame extraction
  numpy                                  OK   1.26.4          Array ops
  Pillow                                 OK   12.2.0          Image processing
  tqdm                                   OK   4.67.3          Progress bars
  retina-face                            OK   0.0.17          Face detection
  imagehash                              OK   4.3.2           pHash dedup
  pybktree                               OK   1.1             BK tree dedup
  scipy                                  OK   1.15.2          Connected components fallback
  requests                    

In [2]:
!pip install -q retina-face imagehash pybktree

In [ ]:
!pip install tf-keras -q

FF++ Real Zip Downloader.

TUM API Setup.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — FF++ REAL VIDEO DOWNLOADER
#
# Downloads the original real videos from FaceForensics++ using the official
# TUM download script. Raw quality, original (real) videos only.
# Fakes are sourced separately from DF40 FS methods — not downloaded here.
#
# What this cell does:
#   1. Validates TUM_LINK is set in environment
#   2. Fetches the TUM download script from the provided URL
#   3. Downloads all 1000 original real videos to local EBS
#   4. Audits the download — file count, total size, sample listing
#   5. Does NOT zip or upload — raw videos stay on EBS for Cell 2
#
# No checkpointing by design. If the download fails, rerun this cell clean.
# The output directory is wiped at the start of each run.
# ─────────────────────────────────────────────────────────────────────────────

import os
import subprocess
import shutil
import glob
import urllib.request
import time
from datetime import datetime, timezone

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR         = "/home/ec2-user/SageMaker/df40_run5"
FFPP_RAW_DIR     = os.path.join(BASE_DIR, "ffpp_raw_real")
DOWNLOAD_SCRIPT  = os.path.join(BASE_DIR, "ffpp_download.py")

TARGET_VIDEOS    = 1000
DOWNLOAD_SERVER  = "EU2"
COMPRESSION      = "raw"

# ─────────────────────────────────────────────────────────────────────────────
# PREFLIGHT
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 70)
print("  CELL 1 — FF++ REAL VIDEO DOWNLOADER")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("=" * 70)

# Confirm TUM_LINK is loaded
TUM_LINK = os.environ.get("TUM_LINK")
if not TUM_LINK:
    raise RuntimeError(
        "FATAL: TUM_LINK environment variable is not set.\n"
        "Run the TUM Link Setup cell first."
    )
print(f"[OK] TUM_LINK found in environment.")

# Wipe and recreate working directories
if os.path.exists(FFPP_RAW_DIR):
    print(f"[INFO] Removing existing output directory: {FFPP_RAW_DIR}")
    shutil.rmtree(FFPP_RAW_DIR)

os.makedirs(BASE_DIR,     exist_ok=True)
os.makedirs(FFPP_RAW_DIR, exist_ok=True)
print(f"[OK] Output directory ready: {FFPP_RAW_DIR}")

# ─────────────────────────────────────────────────────────────────────────────
# FETCH TUM DOWNLOAD SCRIPT
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n[1/3] Fetching TUM download script...")
urllib.request.urlretrieve(TUM_LINK, DOWNLOAD_SCRIPT)

if not os.path.isfile(DOWNLOAD_SCRIPT) or os.path.getsize(DOWNLOAD_SCRIPT) == 0:
    raise RuntimeError(
        "FATAL: TUM download script could not be fetched.\n"
        "Check that TUM_LINK is correct and not expired."
    )
print(f"[OK] Script saved: {DOWNLOAD_SCRIPT} ({os.path.getsize(DOWNLOAD_SCRIPT):,} bytes)")

# ─────────────────────────────────────────────────────────────────────────────
# DOWNLOAD ORIGINAL REAL VIDEOS
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n[2/3] Downloading {TARGET_VIDEOS:,} original real videos...")
print(f"      Server     : {DOWNLOAD_SERVER}")
print(f"      Compression: {COMPRESSION}")
print(f"      Output     : {FFPP_RAW_DIR}")
print()

start_time = time.time()

cmd = (
    f'echo "" | python3 {DOWNLOAD_SCRIPT} {FFPP_RAW_DIR} '
    f'-d original '
    f'-c {COMPRESSION} '
    f'-n {TARGET_VIDEOS} '
    f'--server {DOWNLOAD_SERVER}'
)

from tqdm.auto import tqdm
import re

process = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

pbar = tqdm(total=TARGET_VIDEOS, desc="Downloading FF++ reals", unit="video")
downloaded = 0

for line in process.stdout:
    # TUM script prints one line per video download — detect progress updates
    if "it/s" in line or "it]" in line:
        # Parse current count from tqdm-style output e.g. " 42%| 420/1000"
        match = re.search(r'(\d+)/\d+', line)
        if match:
            current = int(match.group(1))
            pbar.n = current
            pbar.refresh()
    elif any(keyword in line.lower() for keyword in ["error", "fatal", "failed", "traceback"]):
        # Still print errors so you can see them
        tqdm.write(line.rstrip())

process.wait()
pbar.n = TARGET_VIDEOS if process.returncode == 0 else pbar.n
pbar.refresh()
pbar.close()

elapsed = time.time() - start_time

if process.returncode != 0:
    raise RuntimeError(
        f"FATAL: TUM download script exited with code {process.returncode}.\n"
        f"Check output above for details."
    )

print(f"\n[OK] Download process completed in {elapsed:.0f}s ({elapsed/60:.1f} min)")

# ─────────────────────────────────────────────────────────────────────────────
# AUDIT DOWNLOADED FILES
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n[3/3] Auditing downloaded files...")

video_files = glob.glob(os.path.join(FFPP_RAW_DIR, "**", "*.mp4"), recursive=True)
total_size_gb = sum(os.path.getsize(f) for f in video_files) / (1024 ** 3)

print(f"\n  Videos found  : {len(video_files):,}")
print(f"  Total size    : {total_size_gb:.2f} GB")
print(f"  Output dir    : {FFPP_RAW_DIR}")

if len(video_files) == 0:
    raise RuntimeError(
        "FATAL: No .mp4 files found after download.\n"
        "Check credentials and TUM server availability."
    )

if len(video_files) < TARGET_VIDEOS:
    print(f"\n  [WARN] Only {len(video_files):,} of {TARGET_VIDEOS:,} videos downloaded.")
    print(f"         Proceeding — Cell 2 will work with whatever is available.")
else:
    print(f"\n  [OK] Full quota of {TARGET_VIDEOS:,} videos secured.")

# Sample listing
print(f"\n  Sample files:")
for f in sorted(video_files)[:5]:
    size_mb = os.path.getsize(f) / (1024 ** 2)
    print(f"    {os.path.relpath(f, FFPP_RAW_DIR)}  ({size_mb:.1f} MB)")
if len(video_files) > 5:
    print(f"    ... and {len(video_files) - 5:,} more")

# Clean up download script
if os.path.isfile(DOWNLOAD_SCRIPT):
    os.remove(DOWNLOAD_SCRIPT)
    print(f"\n  [Cleanup] TUM script wiped: {DOWNLOAD_SCRIPT}")

# ─────────────────────────────────────────────────────────────────────────────
# DONE
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'=' * 70}")
print("  CELL 1 COMPLETE")
print(f"{'=' * 70}")
print(f"  Videos downloaded : {len(video_files):,}")
print(f"  Total size        : {total_size_gb:.2f} GB")
print(f"  Location          : {FFPP_RAW_DIR}")
print(f"  Elapsed           : {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"  Next              : Run Cell 2 to pull DF40 FS fakes from S3")
print(f"{'=' * 70}")

  CELL 1 — FF++ REAL VIDEO DOWNLOADER
  2026-05-08 16:54:15 UTC
[OK] TUM_LINK found in environment.
[OK] Output directory ready: /home/ec2-user/SageMaker/df40_run5/ffpp_raw_real

[1/3] Fetching TUM download script...
[OK] Script saved: /home/ec2-user/SageMaker/df40_run5/ffpp_download.py (10,727 bytes)

[2/3] Downloading 1,000 original real videos...
      Server     : EU2
      Compression: raw
      Output     : /home/ec2-user/SageMaker/df40_run5/ffpp_raw_real




[OK] Download process completed in 5391s (89.8 min)

[3/3] Auditing downloaded files...

  Videos found  : 1,000
  Total size    : 99.13 GB
  Output dir    : /home/ec2-user/SageMaker/df40_run5/ffpp_raw_real

  [OK] Full quota of 1,000 videos secured.

  Sample files:
    original_sequences/youtube/raw/videos/000.mp4  (54.8 MB)
    original_sequences/youtube/raw/videos/001.mp4  (124.4 MB)
    original_sequences/youtube/raw/videos/002.mp4  (127.4 MB)
    original_sequences/youtube/raw/videos/003.mp4  (24.5 MB)
    original_sequences/youtube/raw/videos/004.mp4  (70.6 MB)
    ... and 995 more

  [Cleanup] TUM script wiped: /home/ec2-user/SageMaker/df40_run5/ffpp_download.py

  CELL 1 COMPLETE
  Videos downloaded : 1,000
  Total size        : 99.13 GB
  Location          : /home/ec2-user/SageMaker/df40_run5/ffpp_raw_real
  Elapsed           : 5391s (89.8 min)
  Next              : Run Cell 2 to pull DF40 FS fakes from S3


Data Loader + Zip Extraction + Folder Structure Audit.

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — DF40 FS FAKE LOADER + UNIFIED EXTRACTION + AUDIT
#
# What this cell does:
#   1. Pulls all 9 DF40 FS method zips from S3 to local EBS
#   2. Extracts all 9 fake zips in one unified extraction phase
#   3. Audits everything together — FF++ reals (from Cell 1) + all 9 DF40 methods
#      Reports per-method video counts, folder structure, total inventory
#
# Resume logic:
#   - S3 pull: skips any zip already present on EBS (size-verified)
#   - Extraction: skips any method folder already extracted and non-empty
#   - Rerun safe — will not re-pull or re-extract what already exists
#
# Output:
#   - All videos unpacked under DATA_POOL_DIR ready for Cell 3
#   - Audit report printed to output and saved to disk
# ─────────────────────────────────────────────────────────────────────────────

import os
import boto3
import zipfile
import shutil
import time
import json
from datetime import datetime, timezone
from tqdm.auto import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR        = "/home/ec2-user/SageMaker/df40_run5"
FFPP_RAW_DIR = "/home/ec2-user/SageMaker/df40_run5/ffpp_raw_real/original_sequences/youtube/raw/videos"       # from Cell 1
ZIP_STAGE_DIR   = os.path.join(BASE_DIR, "df40_zips")           # downloaded zips land here
DATA_POOL_DIR   = os.path.join(BASE_DIR, "data_pool")           # final extracted pool
AUDIT_PATH      = os.path.join(BASE_DIR, "cell2_audit.json")

S3_BUCKET       = "deepfake-d-100k-dataset-tw26"
S3_PREFIX       = "datasets/df40_fs_raw"

DF40_METHODS = [
    "fsgan",
    "faceswap",
    "simswap",
    "inswap",
    "blendface",
    "uniface",
    "mobileswap",
    "e4s",
    "facedancer",
]

# ─────────────────────────────────────────────────────────────────────────────
# PREFLIGHT
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 70)
print("  CELL 2 — DF40 FS FAKE LOADER + UNIFIED EXTRACTION + AUDIT")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("=" * 70)

# Confirm Cell 1 output exists
if not os.path.isdir(FFPP_RAW_DIR):
    raise RuntimeError(
        f"FATAL: FF++ real video directory not found: {FFPP_RAW_DIR}\n"
        "Run Cell 1 first."
    )

real_videos_present = []
for root, dirs, files in os.walk(FFPP_RAW_DIR):
    for f in files:
        if f.endswith(".mp4"):
            real_videos_present.append(os.path.join(root, f))

if len(real_videos_present) == 0:
    raise RuntimeError(
        f"FATAL: No .mp4 files found in {FFPP_RAW_DIR}.\n"
        "Cell 1 may not have completed successfully."
    )

print(f"[OK] Cell 1 output verified: {len(real_videos_present):,} real videos on EBS")

os.makedirs(ZIP_STAGE_DIR, exist_ok=True)
os.makedirs(DATA_POOL_DIR, exist_ok=True)

s3 = boto3.client("s3")

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1 — PULL ALL DF40 FS ZIPS FROM S3
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'=' * 70}")
print("  PHASE 1 — S3 PULL")
print(f"{'=' * 70}")
print(f"  Source  : s3://{S3_BUCKET}/{S3_PREFIX}/")
print(f"  Methods : {len(DF40_METHODS)}")
print(f"  Target  : {ZIP_STAGE_DIR}")
print()

pull_start = time.time()
pull_results = {}

for method in DF40_METHODS:
    s3_key    = f"{S3_PREFIX}/{method}.zip"
    local_zip = os.path.join(ZIP_STAGE_DIR, f"{method}.zip")

    # Resume guard — skip if already present and matches S3 size
    if os.path.isfile(local_zip):
        local_size = os.path.getsize(local_zip)
        try:
            s3_size = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)["ContentLength"]
            if local_size == s3_size:
                print(f"  [SKIP] {method}.zip already on EBS ({local_size / (1024**3):.2f} GB)")
                pull_results[method] = {"status": "skipped", "zip_path": local_zip, "size_gb": local_size / (1024**3)}
                continue
            else:
                print(f"  [WARN] {method}.zip size mismatch — re-downloading")
        except Exception as e:
            print(f"  [WARN] Could not verify {method}.zip on S3: {e} — re-downloading")

    # Pull from S3
    try:
        s3_size = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)["ContentLength"]
    except Exception as e:
        raise RuntimeError(
            f"FATAL: Cannot find s3://{S3_BUCKET}/{s3_key}\n"
            f"Check that the zip was uploaded correctly.\n{e}"
        )

    print(f"  Pulling {method}.zip  ({s3_size / (1024**3):.2f} GB)...")

    with tqdm(total=s3_size, unit="B", unit_scale=True, desc=f"  {method}", leave=False) as pbar:
        def _progress(chunk):
            pbar.update(chunk)
        s3.download_file(
            S3_BUCKET, s3_key, local_zip,
            Callback=_progress
        )

    # Verify
    local_size = os.path.getsize(local_zip)
    if local_size != s3_size:
        raise RuntimeError(
            f"FATAL: Size mismatch after download for {method}.zip\n"
            f"  S3    : {s3_size:,} bytes\n"
            f"  Local : {local_size:,} bytes"
        )

    print(f"  [OK]   {method}.zip  ({local_size / (1024**3):.2f} GB)")
    pull_results[method] = {"status": "pulled", "zip_path": local_zip, "size_gb": local_size / (1024**3)}

pull_elapsed = time.time() - pull_start
print(f"\n  S3 pull complete in {pull_elapsed:.0f}s ({pull_elapsed/60:.1f} min)")

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2 — UNIFIED EXTRACTION (ALL METHODS AT ONCE)
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'=' * 70}")
print("  PHASE 2 — UNIFIED EXTRACTION")
print(f"{'=' * 70}")
print(f"  Extracting {len(DF40_METHODS)} method zips + symlinking FF++ reals")
print(f"  Output : {DATA_POOL_DIR}")
print()

extract_start = time.time()
extract_results = {}

# Extract each DF40 method zip
for method in DF40_METHODS:
    method_out = os.path.join(DATA_POOL_DIR, "fake", method)

    # Resume guard — skip if already extracted and non-empty
    if os.path.isdir(method_out):
        existing = []
        for root, dirs, files in os.walk(method_out):
            for f in files:
                if f.endswith((".mp4", ".avi", ".mov", ".mkv")):
                    existing.append(f)
        if len(existing) > 0:
            print(f"  [SKIP] {method} already extracted ({len(existing):,} videos)")
            extract_results[method] = {"status": "skipped", "videos": len(existing)}
            continue

    os.makedirs(method_out, exist_ok=True)
    zip_path = pull_results[method]["zip_path"]

    print(f"  Extracting {method}.zip ...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        members = zf.namelist()
        for member in tqdm(members, desc=f"  {method}", leave=False):
            zf.extract(member, method_out)

    # Count extracted videos
    extracted_videos = []
    for root, dirs, files in os.walk(method_out):
        for f in files:
            if f.endswith((".mp4", ".avi", ".mov", ".mkv")):
                extracted_videos.append(f)

    print(f"  [OK]   {method} — {len(extracted_videos):,} videos extracted")
    extract_results[method] = {"status": "extracted", "videos": len(extracted_videos)}

# Link FF++ reals into the data pool (copy path reference, no duplication)
real_pool_dir = os.path.join(DATA_POOL_DIR, "real", "ffpp_original")
os.makedirs(real_pool_dir, exist_ok=True)

# Symlink real videos into pool so audit sees them in one place
real_linked = 0
for vid_path in real_videos_present:
    link_path = os.path.join(real_pool_dir, os.path.basename(vid_path))
    if not os.path.exists(link_path):
        os.symlink(vid_path, link_path)
    real_linked += 1

print(f"\n  [OK]   ffpp_original — {real_linked:,} real videos linked into pool")

extract_elapsed = time.time() - extract_start
print(f"\n  Extraction complete in {extract_elapsed:.0f}s ({extract_elapsed/60:.1f} min)")

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3 — UNIFIED AUDIT
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'=' * 70}")
print("  PHASE 3 — UNIFIED AUDIT")
print(f"{'=' * 70}")

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

audit = {
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
    "real": {},
    "fake": {},
    "summary": {}
}

total_real_videos = 0
total_fake_videos = 0

# Audit reals
print(f"\n  REAL SOURCES")
print(f"  {'Source':<20} {'Videos':>8} {'Depth'}")
print(f"  {'-'*20} {'-'*8} {'-'*10}")

real_pool_path = os.path.join(DATA_POOL_DIR, "real")
for source in sorted(os.listdir(real_pool_path)):
    source_path = os.path.join(real_pool_path, source)
    if not os.path.isdir(source_path):
        continue

    videos = []
    max_depth = 0
    for root, dirs, files in os.walk(source_path):
        depth = root.replace(source_path, "").count(os.sep)
        max_depth = max(max_depth, depth)
        for f in files:
            if os.path.splitext(f)[1].lower() in VIDEO_EXTS:
                videos.append(os.path.join(root, f))

    # Sample 5 filenames
    sample_names = [os.path.basename(v) for v in sorted(videos)[:5]]

    audit["real"][source] = {
        "videos": len(videos),
        "max_depth": max_depth,
        "filename_samples": sample_names
    }
    total_real_videos += len(videos)
    print(f"  {source:<20} {len(videos):>8,}   depth={max_depth}")
    print(f"  {'':20}   Sample filenames:")
    for s in sample_names:
        print(f"  {'':20}     {s}")

# Audit fakes
print(f"\n  FAKE SOURCES (DF40 FS METHODS)")
print(f"  {'Method':<20} {'Videos':>8} {'Depth'}")
print(f"  {'-'*20} {'-'*8} {'-'*10}")

fake_pool_path = os.path.join(DATA_POOL_DIR, "fake")
for method in DF40_METHODS:
    method_path = os.path.join(fake_pool_path, method)
    if not os.path.isdir(method_path):
        print(f"  {method:<20} {'MISSING':>8}")
        audit["fake"][method] = {"videos": 0, "max_depth": 0, "warning": "directory_not_found"}
        continue

    videos = []
    max_depth = 0
    subfolder_names = []

    for root, dirs, files in os.walk(method_path):
        depth = root.replace(method_path, "").count(os.sep)
        max_depth = max(max_depth, depth)
        if depth == 1 and len(subfolder_names) < 3:
            subfolder_names.append(os.path.basename(root))
        for f in files:
            if os.path.splitext(f)[1].lower() in VIDEO_EXTS:
                videos.append(os.path.join(root, f))

    # Sample 5 filenames
    sample_names = [os.path.basename(v) for v in sorted(videos)[:5]]

    audit["fake"][method] = {
        "videos": len(videos),
        "max_depth": max_depth,
        "subfolder_sample": subfolder_names,
        "filename_samples": sample_names
    }
    total_fake_videos += len(videos)
    print(f"  {method:<20} {len(videos):>8,}   depth={max_depth}")
    if subfolder_names:
        print(f"  {'':20}   Sample subfolders: {subfolder_names}")
    print(f"  {'':20}   Sample filenames:")
    for s in sample_names:
        print(f"  {'':20}     {s}")

# Summary
print(f"\n{'=' * 70}")
print(f"  AUDIT SUMMARY")
print(f"{'=' * 70}")
print(f"  Total real videos   : {total_real_videos:,}")
print(f"  Total fake videos   : {total_fake_videos:,}")
print(f"  Total video pool    : {total_real_videos + total_fake_videos:,}")
print(f"  DF40 methods found  : {sum(1 for m in DF40_METHODS if audit['fake'].get(m, {}).get('videos', 0) > 0)}/{len(DF40_METHODS)}")

audit["summary"] = {
    "total_real_videos": total_real_videos,
    "total_fake_videos": total_fake_videos,
    "total_videos": total_real_videos + total_fake_videos,
    "methods_with_data": sum(1 for m in DF40_METHODS if audit["fake"].get(m, {}).get("videos", 0) > 0)
}

# Save audit to disk
with open(AUDIT_PATH, "w") as f:
    json.dump(audit, f, indent=2)
print(f"\n  Audit saved: {AUDIT_PATH}")

# Warn if any method is missing
missing_methods = [m for m in DF40_METHODS if audit["fake"].get(m, {}).get("videos", 0) == 0]
if missing_methods:
    print(f"\n  [WARN] Methods with zero videos:")
    for m in missing_methods:
        print(f"    {m}")

if total_real_videos == 0:
    raise RuntimeError("FATAL: Zero real videos in data pool.")
if total_fake_videos == 0:
    raise RuntimeError("FATAL: Zero fake videos in data pool.")

# ─────────────────────────────────────────────────────────────────────────────
# DONE
# ─────────────────────────────────────────────────────────────────────────────

total_elapsed = (pull_elapsed + extract_elapsed)
print(f"\n{'=' * 70}")
print("  CELL 2 COMPLETE")
print(f"{'=' * 70}")
print(f"  S3 pull elapsed     : {pull_elapsed:.0f}s ({pull_elapsed/60:.1f} min)")
print(f"  Extraction elapsed  : {extract_elapsed:.0f}s ({extract_elapsed/60:.1f} min)")
print(f"  Real videos ready   : {total_real_videos:,}")
print(f"  Fake videos ready   : {total_fake_videos:,}")
print(f"  Data pool           : {DATA_POOL_DIR}")
print(f"  Audit report        : {AUDIT_PATH}")
print(f"  Next                : Run Cell 3 — Identity graph + split")
print(f"{'=' * 70}")

  CELL 2 — DF40 FS FAKE LOADER + UNIFIED EXTRACTION + AUDIT
  2026-05-08 18:32:35 UTC
[OK] Cell 1 output verified: 1,000 real videos on EBS

  PHASE 1 — S3 PULL
  Source  : s3://deepfake-d-100k-dataset-tw26/datasets/df40_fs_raw/
  Methods : 9
  Target  : /home/ec2-user/SageMaker/df40_run5/df40_zips

  Pulling fsgan.zip  (3.30 GB)...


  fsgan:   0%|          | 0.00/3.54G [00:00<?, ?B/s]

  [OK]   fsgan.zip  (3.30 GB)
  Pulling faceswap.zip  (1.12 GB)...


  faceswap:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

  [OK]   faceswap.zip  (1.12 GB)
  Pulling simswap.zip  (10.41 GB)...


...

  [OK]   mobileswap — 719 videos extracted
  Extracting e4s.zip ...


  e4s:   0%|          | 0/719 [00:00<?, ?it/s]

  [OK]   e4s — 718 videos extracted
  Extracting facedancer.zip ...


  facedancer:   0%|          | 0/717 [00:00<?, ?it/s]

  [OK]   facedancer — 716 videos extracted

  [OK]   ffpp_original — 1,000 real videos linked into pool

  Extraction complete in 316s (5.3 min)

  PHASE 3 — UNIFIED AUDIT

  REAL SOURCES
  Source                 Videos Depth
  -------------------- -------- ----------
  ffpp_original           1,000   depth=0
                         Sample filenames:
                           000.mp4
                           001.mp4
                           002.mp4
                           003.mp4
                           004.mp4

  FAKE SOURCES (DF40 FS METHODS)
  Method                 Videos Depth
  -------------------- -------- ----------
  fsgan                     686   depth=2
                         Sample subfolders: ['fsgan']
                         Sample filenames:
                           001_870.mp4
                           002_006.mp4
                           005_010.mp4
                           006_002.mp4
                           007_132.mp4
  faceswap            

Zip Cleaup.

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2b — TARGETED CLEANUP
# Deletes all DF40 FS zips (frees ~35GB)
# Deletes uniface extracted folder (incompatible naming, dropped from run)
# Does NOT touch FF++ real videos or any other extracted method folder
# ─────────────────────────────────────────────────────────────────────────────

import os
import shutil

ZIP_STAGE_DIR  = "/home/ec2-user/SageMaker/df40_run5/df40_zips"
DATA_POOL_DIR  = "/home/ec2-user/SageMaker/df40_run5/data_pool"
UNIFACE_DIR    = os.path.join(DATA_POOL_DIR, "fake", "uniface")

print("=" * 60)
print("  CELL 2b — TARGETED CLEANUP")
print("=" * 60)

total_freed = 0

# ── Delete all DF40 zips ──────────────────────────────────────────────────────
print("\n[1/2] Deleting DF40 FS zips...")

if os.path.isdir(ZIP_STAGE_DIR):
    zips = [f for f in os.listdir(ZIP_STAGE_DIR) if f.endswith(".zip")]
    if not zips:
        print("  No zips found.")
    else:
        for z in sorted(zips):
            zip_path = os.path.join(ZIP_STAGE_DIR, z)
            size_gb  = os.path.getsize(zip_path) / (1024 ** 3)
            os.remove(zip_path)
            total_freed += size_gb
            print(f"  Deleted: {z}  ({size_gb:.2f} GB)")
        shutil.rmtree(ZIP_STAGE_DIR, ignore_errors=True)
        print(f"  Zip stage directory removed: {ZIP_STAGE_DIR}")
else:
    print(f"  Zip stage directory not found — already clean.")

# ── Delete uniface extracted folder ──────────────────────────────────────────
print("\n[2/2] Deleting uniface extracted folder...")

if os.path.isdir(UNIFACE_DIR):
    size_gb = sum(
        os.path.getsize(os.path.join(root, f))
        for root, dirs, files in os.walk(UNIFACE_DIR)
        for f in files
    ) / (1024 ** 3)
    shutil.rmtree(UNIFACE_DIR)
    total_freed += size_gb
    print(f"  Deleted: {UNIFACE_DIR}  ({size_gb:.2f} GB)")
else:
    print(f"  Uniface directory not found — already clean.")

# ── Disk space report ─────────────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print(f"  Total freed : {total_freed:.2f} GB")

# Check current disk usage
import shutil as sh
total, used, free = sh.disk_usage("/home/ec2-user/SageMaker")
print(f"  EBS used    : {used / (1024**3):.1f} GB")
print(f"  EBS free    : {free / (1024**3):.1f} GB")
print(f"  EBS total   : {total / (1024**3):.1f} GB")
print(f"{'=' * 60}")
print("  Safe to proceed to Cell 3.")

  CELL 2b — TARGETED CLEANUP

[1/2] Deleting DF40 FS zips...
  Deleted: blendface.zip  (7.71 GB)
  Deleted: e4s.zip  (1.82 GB)
  Deleted: facedancer.zip  (1.19 GB)
  Deleted: faceswap.zip  (1.12 GB)
  Deleted: fsgan.zip  (3.30 GB)
  Deleted: inswap.zip  (3.01 GB)
  Deleted: mobileswap.zip  (3.48 GB)
  Deleted: simswap.zip  (10.41 GB)
  Deleted: uniface.zip  (3.89 GB)
  Zip stage directory removed: /home/ec2-user/SageMaker/df40_run5/df40_zips

[2/2] Deleting uniface extracted folder...
  Deleted: /home/ec2-user/SageMaker/df40_run5/data_pool/fake/uniface  (3.89 GB)

  Total freed : 39.81 GB
  EBS used    : 132.1 GB
  EBS free    : 53.7 GB
  EBS total   : 195.8 GB
  Safe to proceed to Cell 3.


Identity Graph Split to Train/Val/Test.

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — IDENTITY GRAPH + BUCKET ALLOCATION + TRAIN/VAL/TEST SPLIT
#
# Architecture:
#   - Builds unified undirected identity graph from all 8 DF40 FS methods
#   - Computes connected components (BFS)
#   - Allocates components into 4 isolated master buckets (greedy balanced)
#   - Bucket -> split: A+B+C -> train (~70%), D_val -> val (~15%), D_test -> test (~15%)
#   - Strict both-ID same-bucket filter on fakes
#   - Real videos mapped via identity node to correct bucket -> split
#   - Zero identity leakage guaranteed at component level
#   - Saves split_manifest.json for Cell 4
# ══════════════════════════════════════════════════════════════════════════════

import os
import re
import json
import heapq
import random
from collections import defaultdict
from datetime import datetime, timezone

random.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR            = "/home/ec2-user/SageMaker/df40_run5"
DATA_POOL_DIR       = os.path.join(BASE_DIR, "data_pool")
SPLIT_MANIFEST_PATH = os.path.join(BASE_DIR, "split_manifest.json")

FFPP_REAL_DIR = os.path.join(DATA_POOL_DIR, "real", "ffpp_original")
FAKE_POOL_DIR = os.path.join(DATA_POOL_DIR, "fake")

DF40_METHODS = [
    "fsgan", "faceswap", "simswap", "inswap",
    "blendface", "mobileswap", "e4s", "facedancer",
]

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}
SPLITS     = ["train", "val", "test"]

# 4 isolated master buckets
# A+B+C -> train (~70%), D_val -> val (~15%), D_test -> test (~15%)
# D is split internally by proportion after allocation
BUCKETS = ["A", "B", "C", "D_val", "D_test"]
BUCKET_TO_SPLIT = {
    "A"     : "train",
    "B"     : "train",
    "C"     : "train",
    "D_val" : "val",
    "D_test": "test",
}

# Target proportion per bucket for greedy allocator
BUCKET_TARGET_PROPORTION = {
    "A"     : 0.2333,   # ~23.3% -> combined A+B+C = 70%
    "B"     : 0.2333,
    "C"     : 0.2334,
    "D_val" : 0.15,
    "D_test": 0.15,
}

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS — VIDEO DISCOVERY
# ─────────────────────────────────────────────────────────────────────────────

def discover_real_videos(real_dir):
    if not os.path.isdir(real_dir):
        raise RuntimeError(f"Real video directory missing: {real_dir}")
    paths = []
    for root, dirs, files in os.walk(real_dir):
        for f in files:
            if os.path.splitext(f)[1].lower() in VIDEO_EXTS:
                if not f.startswith("._"):
                    paths.append(os.path.join(root, f))
    if not paths:
        raise RuntimeError(f"No video files found under {real_dir}")
    return sorted(paths)


def discover_fake_videos(fake_pool_dir, methods):
    method_paths = {}
    for method in methods:
        method_dir = os.path.join(fake_pool_dir, method)
        if not os.path.isdir(method_dir):
            raise RuntimeError(f"Method directory missing: {method_dir}")
        paths = []
        for root, dirs, files in os.walk(method_dir):
            for f in files:
                ext = os.path.splitext(f)[1].lower()
                if ext in VIDEO_EXTS and not f.startswith("._"):
                    paths.append(os.path.join(root, f))
        if not paths:
            raise RuntimeError(f"No video files found under {method_dir}")
        method_paths[method] = sorted(paths)
    return method_paths

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS — FILENAME PARSING
# ─────────────────────────────────────────────────────────────────────────────

_FAKE_PATTERN = re.compile(r'^(\d+)_(\d+)\.(mp4|avi|mov|mkv)$', re.IGNORECASE)
_REAL_PATTERN = re.compile(r'^(\d+)\.(mp4|avi|mov|mkv)$',       re.IGNORECASE)


def parse_fake_ids(filepath):
    m = _FAKE_PATTERN.match(os.path.basename(filepath))
    return (int(m.group(1)), int(m.group(2))) if m else None


def parse_real_id(filepath):
    m = _REAL_PATTERN.match(os.path.basename(filepath))
    return int(m.group(1)) if m else None

# ─────────────────────────────────────────────────────────────────────────────
# STEP A — BUILD UNIFIED IDENTITY GRAPH
# ─────────────────────────────────────────────────────────────────────────────

def build_identity_graph(method_paths):
    adjacency             = defaultdict(set)
    unique_edges          = set()
    parse_failures        = defaultdict(list)
    raw_pair_observations = 0

    for method, paths in method_paths.items():
        for fp in paths:
            ids = parse_fake_ids(fp)
            if ids is None:
                parse_failures[method].append(fp)
                continue
            id_a, id_b = ids
            adjacency[id_a].add(id_b)
            adjacency[id_b].add(id_a)
            raw_pair_observations += 1
            unique_edges.add(tuple(sorted((id_a, id_b))))

    all_nodes = set(adjacency.keys())
    return dict(adjacency), all_nodes, unique_edges, parse_failures, raw_pair_observations

# ─────────────────────────────────────────────────────────────────────────────
# STEP B — COMPUTE CONNECTED COMPONENTS
# ─────────────────────────────────────────────────────────────────────────────

def compute_connected_components(adjacency, all_nodes):
    visited    = set()
    components = []

    for start in sorted(all_nodes):
        if start in visited:
            continue
        component = set()
        queue     = [start]
        while queue:
            node = queue.pop()
            if node in visited:
                continue
            visited.add(node)
            component.add(node)
            for nb in adjacency.get(node, []):
                if nb not in visited:
                    queue.append(nb)
        components.append(frozenset(component))

    return components

# ─────────────────────────────────────────────────────────────────────────────
# STEP C — ALLOCATE COMPONENTS INTO 5 BUCKETS (70/15/15 TARGET)
# ─────────────────────────────────────────────────────────────────────────────

def allocate_components_to_buckets(components):
    """
    Greedy allocation targeting 70/15/15 split via 5 buckets:
    A+B+C combined = ~70% (train), D_val = ~15% (val), D_test = ~15% (test).
    Each component goes entirely into one bucket — never split.
    """
    total_ids = sum(len(c) for c in components)

    # Target identity count per bucket
    bucket_targets = {
        b: int(total_ids * BUCKET_TARGET_PROPORTION[b])
        for b in BUCKETS
    }

    # Priority queue: (current_deficit, bucket)
    # Deficit = how far below target we still are — fill most-deficit first
    heap = [(-bucket_targets[b], b) for b in BUCKETS]
    heapq.heapify(heap)

    bucket_identities = {b: set() for b in BUCKETS}
    bucket_counts     = {b: 0 for b in BUCKETS}

    sorted_components = sorted(components, key=lambda c: (-len(c), min(c)))

    for comp in sorted_components:
        neg_deficit, bucket = heapq.heappop(heap)
        bucket_identities[bucket].update(comp)
        bucket_counts[bucket] += len(comp)
        # New deficit = target - current count (negative for heap)
        new_deficit = -(bucket_targets[bucket] - bucket_counts[bucket])
        heapq.heappush(heap, (new_deficit, bucket))

    return bucket_identities

# ─────────────────────────────────────────────────────────────────────────────
# STEP D — MAP REAL VIDEOS TO BUCKETS -> SPLITS
# ─────────────────────────────────────────────────────────────────────────────

def map_real_videos(real_paths, bucket_identities):
    node_to_bucket = {}
    for b, id_set in bucket_identities.items():
        for node in id_set:
            node_to_bucket[node] = b

    split_reals = {split: [] for split in SPLITS}
    unmatched   = []

    for fp in real_paths:
        vid_id = parse_real_id(fp)
        if vid_id is None:
            unmatched.append(fp)
            continue
        bucket = node_to_bucket.get(vid_id)
        if bucket is None:
            unmatched.append(fp)
            continue
        split = BUCKET_TO_SPLIT[bucket]
        split_reals[split].append(fp)

    return split_reals, unmatched, node_to_bucket

# ─────────────────────────────────────────────────────────────────────────────
# STEP E — MAP FAKE VIDEOS WITH STRICT BOTH-ID SAME-BUCKET FILTER
# ─────────────────────────────────────────────────────────────────────────────

def map_fake_videos(method_paths, bucket_identities, node_to_bucket):
    split_fakes    = {method: {split: [] for split in SPLITS} for method in method_paths}
    skipped_fakes  = {method: [] for method in method_paths}
    parse_failures = {method: [] for method in method_paths}

    for method, paths in method_paths.items():
        for fp in paths:
            ids = parse_fake_ids(fp)
            if ids is None:
                parse_failures[method].append(fp)
                skipped_fakes[method].append(fp)
                continue

            id_a, id_b = ids
            bucket_a = node_to_bucket.get(id_a)
            bucket_b = node_to_bucket.get(id_b)

            if bucket_a is None or bucket_b is None or bucket_a != bucket_b:
                skipped_fakes[method].append(fp)
                continue

            split = BUCKET_TO_SPLIT[bucket_a]
            split_fakes[method][split].append(fp)

    return split_fakes, skipped_fakes, parse_failures

# ─────────────────────────────────────────────────────────────────────────────
# STEP F — PREFLIGHT AUDIT
# ─────────────────────────────────────────────────────────────────────────────

def print_audit(
    real_paths, method_paths,
    all_nodes, raw_pair_observations, unique_edges, components,
    bucket_identities, node_to_bucket,
    split_reals, unmatched_reals,
    split_fakes, skipped_fakes, parse_failures,
):
    sep  = "═" * 70
    line = "─" * 70
    total_ids = sum(len(ids) for ids in bucket_identities.values())

    print(f"\n{sep}")
    print("  CELL 3 — PREFLIGHT AUDIT REPORT")
    print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print(sep)

    print(f"\n{line}")
    print("  [1] DISCOVERED VIDEOS")
    print(line)
    print(f"  Real videos       : {len(real_paths):>6,}")
    total_fake = sum(len(v) for v in method_paths.values())
    print(f"  Total fake videos : {total_fake:>6,}")
    for method in DF40_METHODS:
        print(f"    {method:<20}: {len(method_paths[method]):>6,}")

    print(f"\n{line}")
    print("  [2] IDENTITY GRAPH")
    print(line)
    print(f"  Nodes (unique IDs)       : {len(all_nodes):>6,}")
    print(f"  Raw pair observations    : {raw_pair_observations:>6,}")
    print(f"  Unique graph edges       : {len(unique_edges):>6,}")
    print(f"  Connected components     : {len(components):>6,}")
    comp_sizes = sorted([len(c) for c in components], reverse=True)
    if comp_sizes:
        print(f"  Largest component        : {comp_sizes[0]:>6,} identities")
        print(f"  Smallest component       : {comp_sizes[-1]:>6,} identities")
        print(f"  Median component         : {comp_sizes[len(comp_sizes)//2]:>6,} identities")
    size_dist = defaultdict(int)
    for s in comp_sizes:
        size_dist[s] += 1
    print(f"  Component size distribution (top 10):")
    for size in sorted(size_dist.keys(), reverse=True)[:10]:
        print(f"    size {size:>4}: {size_dist[size]:>4} components")

    print(f"\n{line}")
    print("  [3] MASTER BUCKET ALLOCATION (5 buckets -> 70/15/15)")
    print(line)
    print(f"  {'Bucket':<10} {'Identities':>12} {'%':>6} {'-> Split'}")
    print(f"  {'-'*10} {'-'*12} {'-'*6} {'-'*10}")
    for b in BUCKETS:
        n   = len(bucket_identities[b])
        pct = n / total_ids * 100 if total_ids else 0
        print(f"  {b:<10} {n:>12,} {pct:>5.1f}%   -> {BUCKET_TO_SPLIT[b]}")

    # Combined split totals
    split_id_counts = defaultdict(int)
    for b, ids in bucket_identities.items():
        split_id_counts[BUCKET_TO_SPLIT[b]] += len(ids)
    print(f"\n  Combined split identity counts:")
    for split in SPLITS:
        pct = split_id_counts[split] / total_ids * 100 if total_ids else 0
        print(f"    {split:<6}: {split_id_counts[split]:>6,} identities  ({pct:.1f}%)")

    print(f"\n{line}")
    print("  [4] REAL VIDEO MAPPING")
    print(line)
    total_real = sum(len(v) for v in split_reals.values())
    for split in SPLITS:
        n   = len(split_reals[split])
        pct = n / total_real * 100 if total_real else 0
        print(f"  {split:<6}: {n:>6,} real videos  ({pct:.1f}%)")
    print(f"  Total mapped  : {total_real:>6,}")
    print(f"  Unmatched     : {len(unmatched_reals):>6,}")
    if unmatched_reals:
        print(f"  [WARN] {len(unmatched_reals):,} real videos could not be mapped")
        for p in unmatched_reals[:5]:
            print(f"    {os.path.basename(p)}")

    print(f"\n{line}")
    print("  [5] FAKE VIDEO MAPPING (strict both-ID same-bucket filter)")
    print(line)
    print(f"  {'Method':<20} {'Train':>8} {'Val':>8} {'Test':>8} "
          f"{'Eligible':>10} {'Skipped':>8} {'Pass%':>7}")
    print(f"  {'-'*20} {'-'*8} {'-'*8} {'-'*8} {'-'*10} {'-'*8} {'-'*7}")

    for method in DF40_METHODS:
        n_train = len(split_fakes[method]["train"])
        n_val   = len(split_fakes[method]["val"])
        n_test  = len(split_fakes[method]["test"])
        n_elig  = n_train + n_val + n_test
        n_skip  = len(skipped_fakes[method])
        n_total = n_elig + n_skip
        pct     = (n_elig / n_total * 100) if n_total else 0
        print(f"  {method:<20} {n_train:>8,} {n_val:>8,} {n_test:>8,} "
              f"{n_elig:>10,} {n_skip:>8,} {pct:>6.1f}%")

    total_elig = sum(len(split_fakes[m][s]) for m in DF40_METHODS for s in SPLITS)
    total_skip = sum(len(skipped_fakes[m]) for m in DF40_METHODS)
    print(f"\n  Total eligible : {total_elig:,}")
    print(f"  Total skipped  : {total_skip:,}")

    total_pf = sum(len(v) for v in parse_failures.values())
    if total_pf:
        print(f"\n{line}")
        print("  [6] PARSE FAILURES")
        print(line)
        for method, fails in parse_failures.items():
            if fails:
                print(f"  {method}: {len(fails)} failures")
                for p in fails[:3]:
                    print(f"    {os.path.basename(p)}")

    print(f"\n{line}")
    print("  [7] WARNINGS")
    print(line)
    warned = False
    for split in SPLITS:
        n = len(split_reals[split])
        if n < 50:
            print(f"  [WARN] {split} has only {n} real videos")
            warned = True
    for method in DF40_METHODS:
        for split in SPLITS:
            n = len(split_fakes[method][split])
            if n < 30:
                print(f"  [WARN] {method}/{split}: only {n} eligible videos")
                warned = True
    if not warned:
        print("  No warnings.")

    print(f"\n{sep}")
    print("  END OF AUDIT REPORT")
    print(f"{sep}\n")

# ─────────────────────────────────────────────────────────────────────────────
# STEP G — FAIL-HARD CHECKS
# ─────────────────────────────────────────────────────────────────────────────

def fail_hard_checks(real_paths, all_nodes, split_reals, split_fakes):
    if not real_paths:
        raise RuntimeError("FATAL: No real videos discovered.")
    if not all_nodes:
        raise RuntimeError("FATAL: Identity graph has zero nodes.")
    for split in SPLITS:
        if not split_reals[split]:
            raise RuntimeError(f"FATAL: No real videos in {split} split.")
    for method in DF40_METHODS:
        total_elig = sum(len(split_fakes[method][s]) for s in SPLITS)
        if total_elig == 0:
            raise RuntimeError(
                f"FATAL: {method} has zero eligible videos after "
                f"strict both-ID same-bucket filtering."
            )
        if not split_fakes[method]["train"]:
            raise RuntimeError(
                f"FATAL: {method} has zero videos in train split."
            )

# ─────────────────────────────────────────────────────────────────────────────
# STEP H — SAVE SPLIT MANIFEST
# ─────────────────────────────────────────────────────────────────────────────

def save_split_manifest(
    split_reals, split_fakes, skipped_fakes,
    bucket_identities, node_to_bucket,
    components, all_nodes, unique_edges,
    raw_pair_observations, unmatched_reals, parse_failures,
):
    split_id_counts = defaultdict(int)
    for b, ids in bucket_identities.items():
        split_id_counts[BUCKET_TO_SPLIT[b]] += len(ids)

    manifest = {
        "generated_at"       : datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
        "bucket_to_split"    : BUCKET_TO_SPLIT,
        "split_ratios_target": {"train": 0.70, "val": 0.15, "test": 0.15},
        "bucket_identities"  : {b: sorted(list(ids)) for b, ids in bucket_identities.items()},
        "real_video_paths"   : {split: sorted(split_reals[split]) for split in SPLITS},
        "fake_video_paths"   : {
            method: {split: sorted(split_fakes[method][split]) for split in SPLITS}
            for method in DF40_METHODS
        },
        "graph_stats": {
            "nodes"                  : len(all_nodes),
            "unique_edges"           : len(unique_edges),
            "raw_pair_observations"  : raw_pair_observations,
            "connected_components"   : len(components),
            "largest_component_size" : max((len(c) for c in components), default=0),
        },
        "split_identity_counts": dict(split_id_counts),
        "split_video_counts": {
            "real": {split: len(split_reals[split]) for split in SPLITS},
            "fake": {
                method: {split: len(split_fakes[method][split]) for split in SPLITS}
                for method in DF40_METHODS
            },
        },
        "audit_stats": {
            "total_real_unmatched" : len(unmatched_reals),
            "total_fake_eligible"  : {
                method: sum(len(split_fakes[method][s]) for s in SPLITS)
                for method in DF40_METHODS
            },
            "total_fake_skipped"   : {method: len(skipped_fakes[method]) for method in DF40_METHODS},
            "parse_failures"       : {method: len(v) for method, v in parse_failures.items()},
        },
    }

    with open(SPLIT_MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2)

    size_kb = os.path.getsize(SPLIT_MANIFEST_PATH) / 1024
    print(f"  Split manifest saved: {SPLIT_MANIFEST_PATH}  ({size_kb:.1f} KB)")

# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("  CELL 3 — IDENTITY GRAPH + BUCKET ALLOCATION + TRAIN/VAL/TEST SPLIT")
print(f"  {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("═" * 70 + "\n")

print("[A] Discovering video files...")
real_paths   = discover_real_videos(FFPP_REAL_DIR)
method_paths = discover_fake_videos(FAKE_POOL_DIR, DF40_METHODS)
print(f"  Real videos : {len(real_paths):,}")
for method in DF40_METHODS:
    print(f"  {method:<20}: {len(method_paths[method]):,} videos")

print("\n[B] Building unified identity graph across all 8 methods...")
adjacency, all_nodes, unique_edges, parse_failures, raw_pair_observations = build_identity_graph(method_paths)
print(f"  Nodes             : {len(all_nodes):,}")
print(f"  Pair observations : {raw_pair_observations:,}")
print(f"  Unique edges      : {len(unique_edges):,}")

print("\n[C] Computing connected components...")
components = compute_connected_components(adjacency, all_nodes)
print(f"  Components        : {len(components):,}")
print(f"  Largest           : {max((len(c) for c in components), default=0):,} identities")

print("\n[D] Allocating components into 5 buckets targeting 70/15/15...")
bucket_identities = allocate_components_to_buckets(components)
total_ids = sum(len(ids) for ids in bucket_identities.values())
for b in BUCKETS:
    n   = len(bucket_identities[b])
    pct = n / total_ids * 100 if total_ids else 0
    print(f"  Bucket {b:<8} -> {BUCKET_TO_SPLIT[b]:<6}: {n:,} identities  ({pct:.1f}%)")

split_id_totals = defaultdict(int)
for b, ids in bucket_identities.items():
    split_id_totals[BUCKET_TO_SPLIT[b]] += len(ids)
print(f"  Combined: train={split_id_totals['train']:,}  val={split_id_totals['val']:,}  test={split_id_totals['test']:,}")

print("\n[E] Mapping real videos to buckets -> splits...")
split_reals, unmatched_reals, node_to_bucket = map_real_videos(real_paths, bucket_identities)
total_real = sum(len(v) for v in split_reals.values())
for split in SPLITS:
    pct = len(split_reals[split]) / total_real * 100 if total_real else 0
    print(f"  {split:<6}: {len(split_reals[split]):,} real videos  ({pct:.1f}%)")
if unmatched_reals:
    print(f"  [WARN] {len(unmatched_reals):,} unmatched real videos")

print("\n[F] Mapping fake videos with strict both-ID same-bucket filter...")
split_fakes, skipped_fakes, fake_parse_failures = map_fake_videos(
    method_paths, bucket_identities, node_to_bucket
)
for method in DF40_METHODS:
    total_elig = sum(len(split_fakes[method][s]) for s in SPLITS)
    total_skip = len(skipped_fakes[method])
    print(f"  {method:<20}: {total_elig:,} eligible / {total_skip:,} skipped")

print("\n[G] Running preflight audit...")
print_audit(
    real_paths, method_paths,
    all_nodes, raw_pair_observations, unique_edges, components,
    bucket_identities, node_to_bucket,
    split_reals, unmatched_reals,
    split_fakes, skipped_fakes, fake_parse_failures,
)

print("[H] Fail-hard validation checks...")
fail_hard_checks(real_paths, all_nodes, split_reals, split_fakes)
print("  All fail-hard checks passed.\n")

print("[I] Saving split manifest...")
save_split_manifest(
    split_reals, split_fakes, skipped_fakes,
    bucket_identities, node_to_bucket,
    components, all_nodes, unique_edges,
    raw_pair_observations, unmatched_reals, fake_parse_failures,
)

print("\n" + "─" * 70)
print("  CELL 3 COMPLETE")
print(f"  Split manifest : {SPLIT_MANIFEST_PATH}")
print("  Next           : Run Cell 4 — Frame extraction")
print("─" * 70 + "\n")


══════════════════════════════════════════════════════════════════════
  CELL 3 — IDENTITY GRAPH + BUCKET ALLOCATION + TRAIN/VAL/TEST SPLIT
  2026-05-08 20:34:07 UTC
══════════════════════════════════════════════════════════════════════

[A] Discovering video files...
  Real videos : 1,000
  fsgan               : 686 videos
  faceswap            : 720 videos
  simswap             : 989 videos
  inswap              : 883 videos
  blendface           : 712 videos
  mobileswap          : 719 videos
  e4s                 : 718 videos
  facedancer          : 716 videos

[B] Building unified identity graph across all 8 methods...
  Nodes             : 1,000
  Pair observations : 6,143
  Unique edges      : 500

[C] Computing connected components...
  Components        : 500
  Largest           : 2 identities

[D] Allocating components into 5 buckets targeting 70/15/15...
  Bucket A        -> train : 234 identities  (23.4%)
  Bucket B        -> train : 234 identities  (23.4%)
  Bucket C     

Frame Extraction Engine.

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — FRAME EXTRACTION ENGINE
#
# Reads split_manifest.json from Cell 3.
# Extracts frames from all real and fake videos per split.
# Shuffles video order before extraction for diversity.
# Saves raw full-resolution JPG frames into split/class/method directory tree.
# Zips and uploads to S3 with byte-for-byte verification.
# Deletes FF++ raw videos after real extraction to free ~99GB EBS.
# Cleans local frames only after verified S3 upload.
#
# Frame counts:
#   Real  : 30 evenly-spaced frames per video (np.linspace)
#   Fake  : 10 evenly-spaced frames per video (np.linspace)
#
# Resume logic:
#   - Checks S3 for existing verified upload before re-extracting
#   - If S3 zip already present and size-verified, skips extraction entirely
# ══════════════════════════════════════════════════════════════════════════════

import os
import json
import cv2
import csv
import shutil
import zipfile
import boto3
import time
import random
import numpy as np
from datetime import datetime, timezone
from tqdm.auto import tqdm

random.seed(42)
np.random.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR            = "/home/ec2-user/SageMaker/df40_run5"
SPLIT_MANIFEST_PATH = os.path.join(BASE_DIR, "split_manifest.json")
FRAMES_DIR          = os.path.join(BASE_DIR, "extracted_frames")
LOGS_DIR            = os.path.join(BASE_DIR, "logs")
ARCHIVE_PATH        = os.path.join(BASE_DIR, "df40_run5_raw_frames.zip")
LOG_CSV_PATH        = os.path.join(LOGS_DIR, "cell4_frame_extraction_log.csv")
MANIFEST_PATH       = os.path.join(LOGS_DIR, "cell4_manifest.txt")

FFPP_RAW_DIR        = "/home/ec2-user/SageMaker/df40_run5/ffpp_raw_real"

S3_BUCKET           = "deepfake-d-100k-dataset-tw26"
S3_PREFIX           = "datasets/df40_run5/raw_frames"
S3_ARCHIVE_KEY      = f"{S3_PREFIX}/df40_run5_raw_frames.zip"
S3_LOG_KEY          = f"{S3_PREFIX}/cell4_frame_extraction_log.csv"
S3_MANIFEST_KEY     = f"{S3_PREFIX}/cell4_manifest.txt"

REAL_FRAMES_PER_VIDEO = 30
FAKE_FRAMES_PER_VIDEO = 10

SPLITS   = ["train", "val", "test"]
DF40_METHODS = [
    "fsgan", "faceswap", "simswap", "inswap",
    "blendface", "mobileswap", "e4s", "facedancer",
]

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS — FRAME INDICES
# ─────────────────────────────────────────────────────────────────────────────

def compute_frame_indices(total_frames, target_count):
    """
    Evenly spaced frame indices using np.linspace.
    If total_frames <= target_count returns all indices.
    Deduplicates while preserving order.
    """
    if total_frames <= 0:
        return []
    if total_frames <= target_count:
        return list(range(total_frames))
    indices = np.linspace(0, total_frames - 1, num=target_count, dtype=int)
    seen   = set()
    unique = []
    for idx in indices:
        if idx not in seen:
            seen.add(idx)
            unique.append(int(idx))
    return unique

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS — SINGLE VIDEO EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────

def extract_frames_from_video(video_path, output_dir, target_count, video_counter):
    """
    Extract evenly-spaced frames from a single video.
    Saves as JPG quality 95.
    Returns result dict for CSV log.
    """
    result = {
        "video_path"           : video_path,
        "output_dir"           : output_dir,
        "total_frames_metadata": 0,
        "target_frames"        : target_count,
        "frames_saved"         : 0,
        "status"               : "failed",
        "failure_reason"       : "",
    }

    video_stem = os.path.splitext(os.path.basename(video_path))[0]

    if not os.path.isfile(video_path):
        result["failure_reason"] = "file_not_found"
        return result

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        result["failure_reason"] = "cv2_cannot_open"
        return result

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    result["total_frames_metadata"] = total_frames

    if total_frames <= 0:
        cap.release()
        result["failure_reason"] = "zero_frame_count"
        return result

    indices = compute_frame_indices(total_frames, target_count)
    if not indices:
        cap.release()
        result["failure_reason"] = "no_valid_indices"
        return result

    os.makedirs(output_dir, exist_ok=True)
    saved = 0

    for frame_idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret or frame is None:
            continue
        fname    = f"{video_counter:05d}_{video_stem}_f{frame_idx:06d}.jpg"
        out_path = os.path.join(output_dir, fname)
        try:
            cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved += 1
        except Exception as e:
            tqdm.write(f"  [WARN] Failed to write frame {frame_idx} for {video_stem}: {e}")

    cap.release()
    result["frames_saved"] = saved

    if saved == 0:
        result["status"]         = "failed"
        result["failure_reason"] = "all_frame_reads_failed"
    elif saved < len(indices):
        result["status"]         = "partial"
        result["failure_reason"] = f"only_{saved}_of_{len(indices)}_frames_saved"
    else:
        result["status"] = "success"

    return result

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS — S3
# ─────────────────────────────────────────────────────────────────────────────

def upload_and_verify(s3, local_path, bucket, s3_key):
    local_size = os.path.getsize(local_path)
    size_gb    = local_size / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)")
    print(f"    -> s3://{bucket}/{s3_key}")
    s3.upload_file(local_path, bucket, s3_key)
    remote_size = s3.head_object(Bucket=bucket, Key=s3_key)["ContentLength"]
    if remote_size != local_size:
        raise RuntimeError(
            f"UPLOAD VERIFICATION FAILED for {s3_key}\n"
            f"  Local : {local_size:,} bytes\n"
            f"  S3    : {remote_size:,} bytes"
        )
    print(f"  Verified: {remote_size:,} bytes match.")


def check_s3_resume(s3, local_path, bucket, s3_key):
    """Returns True if S3 object exists and matches local size — safe to skip."""
    if not os.path.isfile(local_path):
        return False
    try:
        remote_size = s3.head_object(Bucket=bucket, Key=s3_key)["ContentLength"]
        local_size  = os.path.getsize(local_path)
        return remote_size == local_size
    except Exception:
        return False

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

start_time    = time.time()
timestamp_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
s3            = boto3.client("s3")

print("=" * 70)
print("  CELL 4 — FRAME EXTRACTION ENGINE")
print(f"  {timestamp_str}")
print(f"  Real frames/video : {REAL_FRAMES_PER_VIDEO}")
print(f"  Fake frames/video : {FAKE_FRAMES_PER_VIDEO}")
print("=" * 70)

# ─── RESUME CHECK ────────────────────────────────────────────────────────────

if check_s3_resume(s3, ARCHIVE_PATH, S3_BUCKET, S3_ARCHIVE_KEY):
    print("\n[RESUME] S3 archive already verified. Cell 4 previously completed.")
    print(f"  s3://{S3_BUCKET}/{S3_ARCHIVE_KEY}")
    print("  Skip to Cell 5.")
    raise SystemExit(0)

# ─── LOAD SPLIT MANIFEST ─────────────────────────────────────────────────────

print("\n[1/8] Loading split manifest...")
if not os.path.isfile(SPLIT_MANIFEST_PATH):
    raise FileNotFoundError(
        f"FATAL: split_manifest.json not found at {SPLIT_MANIFEST_PATH}\n"
        "Run Cell 3 first."
    )

with open(SPLIT_MANIFEST_PATH, "r") as f:
    manifest = json.load(f)

real_video_paths = manifest["real_video_paths"]
fake_video_paths = manifest["fake_video_paths"]

total_real_videos = sum(len(v) for v in real_video_paths.values())
total_fake_videos = sum(
    len(fake_video_paths[m][s])
    for m in DF40_METHODS for s in SPLITS
)

print(f"  Real videos in manifest : {total_real_videos:,}")
print(f"  Fake videos in manifest : {total_fake_videos:,}")
print(f"  Expected real frames    : {total_real_videos * REAL_FRAMES_PER_VIDEO:,}")
print(f"  Expected fake frames    : {total_fake_videos * FAKE_FRAMES_PER_VIDEO:,}")

# ─── PREPARE DIRECTORIES ─────────────────────────────────────────────────────

print("\n[2/8] Preparing output directories...")
if os.path.exists(FRAMES_DIR):
    shutil.rmtree(FRAMES_DIR)
os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(LOGS_DIR,   exist_ok=True)

# Pre-create all split/class/method directories
for split in SPLITS:
    os.makedirs(os.path.join(FRAMES_DIR, split, "real"), exist_ok=True)
    for method in DF40_METHODS:
        os.makedirs(os.path.join(FRAMES_DIR, split, "fake", method), exist_ok=True)

print(f"  Output root: {FRAMES_DIR}")

# ─── EXTRACT REAL FRAMES ─────────────────────────────────────────────────────

print("\n[3/8] Extracting real frames (30 per video)...")

csv_rows      = []
video_counter = 0
real_stats    = {split: {"attempted": 0, "frames": 0, "failed": 0} for split in SPLITS}

for split in SPLITS:
    paths = real_video_paths.get(split, [])
    # Shuffle for diversity
    paths = list(paths)
    random.shuffle(paths)

    out_dir = os.path.join(FRAMES_DIR, split, "real")
    print(f"\n  {split} — {len(paths):,} real videos")

    pbar = tqdm(paths, desc=f"  real/{split}", unit="video", leave=True)
    for vpath in pbar:
        video_counter += 1
        result = extract_frames_from_video(
            video_path    = vpath,
            output_dir    = out_dir,
            target_count  = REAL_FRAMES_PER_VIDEO,
            video_counter = video_counter,
        )
        result["split"]    = split
        result["class"]    = "real"
        result["method"]   = "ffpp_original"
        csv_rows.append(result)

        real_stats[split]["attempted"] += 1
        real_stats[split]["frames"]    += result["frames_saved"]
        if result["status"] == "failed":
            real_stats[split]["failed"] += 1
            tqdm.write(f"  [FAIL] {os.path.basename(vpath)}: {result['failure_reason']}")

total_real_frames = sum(s["frames"] for s in real_stats.values())
print(f"\n  Real extraction complete")
print(f"  {'Split':<8} {'Videos':>8} {'Frames':>10} {'Failed':>8}")
print(f"  {'-'*8} {'-'*8} {'-'*10} {'-'*8}")
for split in SPLITS:
    s = real_stats[split]
    print(f"  {split:<8} {s['attempted']:>8,} {s['frames']:>10,} {s['failed']:>8,}")
print(f"  Total real frames extracted: {total_real_frames:,}")

# ─── DELETE FF++ RAW VIDEOS — FREE ~99GB ─────────────────────────────────────

print("\n[4/8] Deleting FF++ raw videos to free EBS space...")
if os.path.isdir(FFPP_RAW_DIR):
    freed_gb = sum(
        os.path.getsize(os.path.join(root, f))
        for root, dirs, files in os.walk(FFPP_RAW_DIR)
        for f in files
    ) / (1024 ** 3)
    shutil.rmtree(FFPP_RAW_DIR)
    print(f"  Deleted: {FFPP_RAW_DIR}  (~{freed_gb:.1f} GB freed)")
else:
    print(f"  [INFO] {FFPP_RAW_DIR} not found — already deleted.")

# Disk check after deletion
total, used, free = shutil.disk_usage(BASE_DIR)
print(f"  EBS free after deletion: {free / (1024**3):.1f} GB")

# ─── EXTRACT FAKE FRAMES ─────────────────────────────────────────────────────

print("\n[5/8] Extracting fake frames (10 per video)...")

fake_stats = {
    method: {split: {"attempted": 0, "frames": 0, "failed": 0}
             for split in SPLITS}
    for method in DF40_METHODS
}

for method in DF40_METHODS:
    print(f"\n  Method: {method}")
    for split in SPLITS:
        paths = fake_video_paths.get(method, {}).get(split, [])
        paths = list(paths)
        random.shuffle(paths)

        out_dir = os.path.join(FRAMES_DIR, split, "fake", method)
        pbar    = tqdm(paths, desc=f"  {method}/{split}", unit="video", leave=False)

        for vpath in pbar:
            video_counter += 1
            result = extract_frames_from_video(
                video_path    = vpath,
                output_dir    = out_dir,
                target_count  = FAKE_FRAMES_PER_VIDEO,
                video_counter = video_counter,
            )
            result["split"]  = split
            result["class"]  = "fake"
            result["method"] = method
            csv_rows.append(result)

            fake_stats[method][split]["attempted"] += 1
            fake_stats[method][split]["frames"]    += result["frames_saved"]
            if result["status"] == "failed":
                fake_stats[method][split]["failed"] += 1
                tqdm.write(
                    f"  [FAIL] {method}/{split} "
                    f"{os.path.basename(vpath)}: {result['failure_reason']}"
                )

    method_total_frames = sum(
        fake_stats[method][s]["frames"] for s in SPLITS
    )
    method_total_failed = sum(
        fake_stats[method][s]["failed"] for s in SPLITS
    )
    print(f"  {method:<20}: {method_total_frames:,} frames extracted "
          f"({method_total_failed} failed videos)")

total_fake_frames = sum(
    fake_stats[m][s]["frames"]
    for m in DF40_METHODS for s in SPLITS
)
print(f"\n  Total fake frames extracted: {total_fake_frames:,}")

# ─── WRITE CSV LOG ────────────────────────────────────────────────────────────

print("\n[6/8] Writing extraction log...")
csv_fields = [
    "video_path", "split", "class", "method", "output_dir",
    "total_frames_metadata", "target_frames", "frames_saved",
    "status", "failure_reason",
]
with open(LOG_CSV_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields, extrasaction="ignore")
    writer.writeheader()
    for row in csv_rows:
        writer.writerow(row)
print(f"  Rows written: {len(csv_rows):,}")

# ─── WRITE MANIFEST ──────────────────────────────────────────────────────────

print(f"\nWriting manifest...")
elapsed = time.time() - start_time

total_frames = total_real_frames + total_fake_frames
total_failed = sum(
    1 for r in csv_rows if r["status"] == "failed"
)

lines = [
    "=" * 70,
    "  CELL 4 — FRAME EXTRACTION MANIFEST",
    "=" * 70,
    f"  Timestamp              : {timestamp_str}",
    f"  Elapsed                : {elapsed:.0f}s ({elapsed/60:.1f} min)",
    f"  Real frames/video      : {REAL_FRAMES_PER_VIDEO}",
    f"  Fake frames/video      : {FAKE_FRAMES_PER_VIDEO}",
    "",
    "  Real frames by split:",
]
for split in SPLITS:
    s = real_stats[split]
    lines.append(f"    {split:<6}: {s['attempted']:,} videos -> {s['frames']:,} frames")
lines += [
    f"  Total real frames      : {total_real_frames:,}",
    "",
    "  Fake frames by method:",
]
for method in DF40_METHODS:
    mf = sum(fake_stats[method][s]["frames"] for s in SPLITS)
    mv = sum(fake_stats[method][s]["attempted"] for s in SPLITS)
    lines.append(f"    {method:<20}: {mv:,} videos -> {mf:,} frames")
lines += [
    f"  Total fake frames      : {total_fake_frames:,}",
    f"  Total frames           : {total_frames:,}",
    f"  Total failed videos    : {total_failed:,}",
    "",
    "  S3 destinations:",
    f"    Archive  : s3://{S3_BUCKET}/{S3_ARCHIVE_KEY}",
    f"    CSV log  : s3://{S3_BUCKET}/{S3_LOG_KEY}",
    f"    Manifest : s3://{S3_BUCKET}/{S3_MANIFEST_KEY}",
    "=" * 70,
]
with open(MANIFEST_PATH, "w") as f:
    f.write("\n".join(lines) + "\n")
print(f"  Manifest written: {MANIFEST_PATH}")

# ─── ZIP FRAMES ──────────────────────────────────────────────────────────────

print("\n[7/8] Creating zip archive...")

# Disk check before zipping
total, used, free = shutil.disk_usage(BASE_DIR)
frames_size = sum(
    os.path.getsize(os.path.join(root, f))
    for root, dirs, files in os.walk(FRAMES_DIR)
    for f in files
)
print(f"  Frames size : {frames_size / (1024**3):.2f} GB")
print(f"  EBS free    : {free / (1024**3):.2f} GB")

if free < frames_size * 1.1:
    raise RuntimeError(
        f"Insufficient disk space for zip.\n"
        f"  Frames : {frames_size / (1024**3):.2f} GB\n"
        f"  Free   : {free / (1024**3):.2f} GB\n"
        f"Need at least 1.1x frame size free."
    )

file_count = 0
zip_start  = time.time()

with zipfile.ZipFile(ARCHIVE_PATH, "w", zipfile.ZIP_STORED) as zf:
    for root, dirs, files in os.walk(FRAMES_DIR):
        for fname in files:
            abs_path = os.path.join(root, fname)
            arc_name = os.path.relpath(abs_path, FRAMES_DIR)
            zf.write(abs_path, arc_name)
            file_count += 1
            if file_count % 5000 == 0:
                print(f"  ... archived {file_count:,} files so far ...")

zip_elapsed  = time.time() - zip_start
archive_size = os.path.getsize(ARCHIVE_PATH)
print(f"  Files archived : {file_count:,}")
print(f"  Archive size   : {archive_size / (1024**3):.2f} GB")
print(f"  Zip elapsed    : {zip_elapsed:.0f}s")

if not os.path.isfile(ARCHIVE_PATH) or archive_size == 0:
    raise RuntimeError("FATAL: Zip archive is missing or empty.")

# ─── UPLOAD TO S3 ────────────────────────────────────────────────────────────

print("\n[8/8] Uploading to S3...")

upload_and_verify(s3, ARCHIVE_PATH,  S3_BUCKET, S3_ARCHIVE_KEY)
upload_and_verify(s3, LOG_CSV_PATH,  S3_BUCKET, S3_LOG_KEY)
upload_and_verify(s3, MANIFEST_PATH, S3_BUCKET, S3_MANIFEST_KEY)

print("\n  All uploads verified.")

# ─── CLEANUP LOCAL FRAMES + ARCHIVE ──────────────────────────────────────────

print("\n  Cleaning local frames and archive...")
shutil.rmtree(FRAMES_DIR)
os.remove(ARCHIVE_PATH)
print(f"  Removed: {FRAMES_DIR}")
print(f"  Removed: {ARCHIVE_PATH}")
print(f"  Logs preserved: {LOGS_DIR}")

# Final disk check
total, used, free = shutil.disk_usage(BASE_DIR)
print(f"  EBS free after cleanup: {free / (1024**3):.1f} GB")

# ─── DONE ─────────────────────────────────────────────────────────────────────

total_elapsed = time.time() - start_time
print(f"\n{'=' * 70}")
print("  CELL 4 COMPLETE")
print(f"{'=' * 70}")
print(f"  Total videos processed : {video_counter:,}")
print(f"  Total real frames      : {total_real_frames:,}")
print(f"  Total fake frames      : {total_fake_frames:,}")
print(f"  Total frames           : {total_real_frames + total_fake_frames:,}")
print(f"  Failed videos          : {total_failed:,}")
print(f"  Elapsed                : {total_elapsed:.0f}s ({total_elapsed/60:.1f} min)")
print(f"  S3 archive             : s3://{S3_BUCKET}/{S3_ARCHIVE_KEY}")
print(f"  Next                   : Run Cell 5 — RetinaFace")
print(f"{'=' * 70}")

  CELL 4 — FRAME EXTRACTION ENGINE
  2026-05-08 20:39:37 UTC
  Real frames/video : 30
  Fake frames/video : 10

[1/8] Loading split manifest...
  Real videos in manifest : 1,000
  Fake videos in manifest : 6,143
  Expected real frames    : 30,000
  Expected fake frames    : 61,430

[2/8] Preparing output directories...
  Output root: /home/ec2-user/SageMaker/df40_run5/extracted_frames

[3/8] Extracting real frames (30 per video)...

  train — 700 real videos


  real/train:   0%|          | 0/700 [00:00<?, ?video/s]


  val — 150 real videos


  real/val:   0%|          | 0/150 [00:00<?, ?video/s]


  test — 150 real videos


...

  e4s                 : 7,180 frames extracted (0 failed videos)

  Method: facedancer


  facedancer/train:   0%|          | 0/497 [00:00<?, ?video/s]

  facedancer/val:   0%|          | 0/111 [00:00<?, ?video/s]

  facedancer/test:   0%|          | 0/108 [00:00<?, ?video/s]

  facedancer          : 7,160 frames extracted (0 failed videos)

  Total fake frames extracted: 61,430

[6/8] Writing extraction log...
  Rows written: 7,143

Writing manifest...
  Manifest written: /home/ec2-user/SageMaker/df40_run5/logs/cell4_manifest.txt

[7/8] Creating zip archive...
  Frames size : 11.41 GB
  EBS free    : 141.24 GB
  ... archived 5,000 files so far ...
  ... archived 10,000 files so far ...
  ... archived 15,000 files so far ...
  ... archived 20,000 files so far ...
  ... archived 25,000 files so far ...
  ... archived 30,000 files so far ...
  ... archived 35,000 files so far ...
  ... archived 40,000 files so far ...
  ... archived 45,000 files so far ...
  ... archived 50,000 files so far ...
  ... archived 55,000 files so far ...
  ... archived 60,000 files so far ...
  ... archived 65,000 files so far ...
  ... archived 70,000 files so far ...
  ... archived 75,000 files so far ...
  ... archived 80,000 files so far ...
  ... archived 85,000 files so far .

RetinaFace Engine + S3 Upload.

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — RETINAFACE FACE EXTRACTION ENGINE
#
# Architecture:
#   1.  Download raw frames zip from S3 (Cell 4 output)
#   2.  Unpack via native subprocess unzip
#   3.  Validate unpacked folder structure
#   4.  Scan frame paths per split / class / method
#   5.  Process real frames first — shuffle for diversity, fill to REAL_CAP
#   6.  Compute fake quotas dynamically based on actual real yield
#       Distributed proportionally across 8 methods by video count
#   7.  Process fake frames per method — shuffle, fill to proportional quota
#   8.  RetinaFace crop engine: 1080p bypass, confidence filter, blur check,
#       pHash dedup per split-class BK tree, 260x260 bicubic resize
#   9.  Fail hard if any quota bucket underfilled before archive
#  10.  Zip final accepted faces
#  11.  Write CSV log + manifest
#  12.  Upload zip + manifest + CSV to S3 with byte-for-byte verification
#  13.  Clean local staging only after verified upload
#
# Zero leakage: split assignment was locked in Cell 3.
# Diversity guarantee: frame lists shuffled per method before quota filling.
# Dedup isolation: one BK tree per (split x class) — real and fake trees
#   are independent, and val/test trees are independent from train.
# ══════════════════════════════════════════════════════════════════════════════

import csv
import os
os.environ["TF_USE_LEGACY_KERAS"]    = "1"
os.environ["TF_CUDNN_USE_AUTOTUNE"]  = "0"
import random
import shutil
import subprocess
import time
import warnings
from collections import defaultdict
from datetime import datetime, timezone

import boto3
import cv2
import imagehash
import numpy as np
import pybktree
from PIL import Image
from retinaface import RetinaFace
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR         = "/home/ec2-user/SageMaker/df40_run5"
STAGE_DIR        = os.path.join(BASE_DIR, "retinaface_stage")
RAW_ZIP_DIR      = os.path.join(STAGE_DIR, "raw_zip")
RAW_FRAMES_DIR   = os.path.join(STAGE_DIR, "raw_frames")
FINAL_FACES_DIR  = os.path.join(STAGE_DIR, "final_faces")
LOGS_DIR         = os.path.join(BASE_DIR,  "logs")

LOCAL_ZIP_IN     = os.path.join(RAW_ZIP_DIR, "df40_run5_raw_frames.zip")
FINAL_ZIP_BASE   = os.path.join(BASE_DIR,    "df40_run5_final_faces")
CSV_LOG_PATH     = os.path.join(LOGS_DIR,    "cell5_retinaface_log.csv")
MANIFEST_PATH    = os.path.join(LOGS_DIR,    "cell5_manifest.txt")

S3_BUCKET        = "deepfake-d-100k-dataset-tw26"
S3_INPUT_KEY     = "datasets/df40_run5/raw_frames/df40_run5_raw_frames.zip"
S3_OUTPUT_PREFIX = "datasets/df40_run5/final"

S3_OUT_KEYS = {
    "zip"      : f"{S3_OUTPUT_PREFIX}/df40_run5_final_faces.zip",
    "manifest" : f"{S3_OUTPUT_PREFIX}/cell5_manifest.txt",
    "csv"      : f"{S3_OUTPUT_PREFIX}/cell5_retinaface_log.csv",
}

# RetinaFace quality thresholds — validated from previous runs
CONFIDENCE_THRESHOLD = 0.90
MIN_FACE_SIZE        = 30
MIN_FACE_RATIO       = 0.005
BLUR_THRESHOLD       = 25
PHASH_HAMMING_MAX    = 2
PADDING              = 25
RESIZE_TARGET        = (260, 260)

# Real cap — Cell 5 fills up to this many real faces total across all splits
# Fake quotas are computed dynamically after reals complete
REAL_CAP = 22000

# Proportional weights per method based on Cell 3 audit video counts
METHOD_WEIGHTS = {
    "fsgan"      : 686,
    "faceswap"   : 720,
    "simswap"    : 989,
    "inswap"     : 883,
    "blendface"  : 712,
    "mobileswap" : 719,
    "e4s"        : 718,
    "facedancer" : 716,
}
TOTAL_FAKE_VIDEOS = sum(METHOD_WEIGHTS.values())   # 6,143

SPLITS   = ["train", "val", "test"]
SPLIT_PROPORTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}

DF40_METHODS = list(METHOD_WEIGHTS.keys())

s3 = boto3.client("s3")

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS — QUOTA COMPUTATION
# ─────────────────────────────────────────────────────────────────────────────

def compute_real_quotas(real_cap):
    """
    Distribute real_cap across splits at 70/15/15.
    Returns dict: quota_key -> target count.
    """
    quotas = {}
    for split, prop in SPLIT_PROPORTIONS.items():
        quotas[f"{split}_real"] = int(real_cap * prop)
    # Give any rounding remainder to train
    assigned = sum(quotas.values())
    quotas["train_real"] += (real_cap - assigned)
    return quotas


def compute_fake_quotas(fake_total):
    """
    Distribute fake_total across 8 methods proportionally by video count,
    then split each method's total at 70/15/15 across splits.
    Returns dict: quota_key -> target count.
    """
    quotas     = {}
    method_totals = {}

    # Proportional per method
    allocated = 0
    methods_list = DF40_METHODS[:]
    for i, method in enumerate(methods_list):
        if i == len(methods_list) - 1:
            # Last method gets remainder to avoid rounding loss
            method_total = fake_total - allocated
        else:
            method_total = int(fake_total * METHOD_WEIGHTS[method] / TOTAL_FAKE_VIDEOS)
        method_totals[method] = method_total
        allocated += method_total

    # Split each method total at 70/15/15
    for method, total in method_totals.items():
        for split, prop in SPLIT_PROPORTIONS.items():
            quotas[f"{split}_fake_{method}"] = int(total * prop)
        # Rounding remainder to train
        assigned = sum(quotas[f"{s}_fake_{method}"] for s in SPLITS)
        quotas[f"train_fake_{method}"] += (total - assigned)

    return quotas, method_totals


def quota_key_real(split):
    return f"{split}_real"


def quota_key_fake(split, method):
    return f"{split}_fake_{method}"


def all_real_quotas_filled(accepted, real_quotas):
    return all(accepted.get(k, 0) >= real_quotas[k] for k in real_quotas)


def all_fake_quotas_filled(accepted, fake_quotas):
    return all(accepted.get(k, 0) >= fake_quotas[k] for k in fake_quotas)

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS — S3
# ─────────────────────────────────────────────────────────────────────────────

def download_from_s3(s3_key, local_path):
    size_obj = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)["ContentLength"]
    size_gb  = size_obj / (1024 ** 3)
    print(f"  Downloading s3://{S3_BUCKET}/{s3_key}  ({size_gb:.2f} GB)...")
    s3.download_file(S3_BUCKET, s3_key, local_path)
    local_size = os.path.getsize(local_path)
    if local_size != size_obj:
        raise RuntimeError(
            f"Download size mismatch.\n"
            f"  Expected : {size_obj:,} bytes\n"
            f"  Got      : {local_size:,} bytes"
        )
    print(f"  Saved: {local_path}  ({local_size / (1024**3):.2f} GB)")


def upload_and_verify(local_path, s3_key):
    local_size = os.path.getsize(local_path)
    size_gb    = local_size / (1024 ** 3)
    print(f"  Uploading {os.path.basename(local_path)}  ({size_gb:.2f} GB)")
    print(f"    -> s3://{S3_BUCKET}/{s3_key}")
    s3.upload_file(local_path, S3_BUCKET, s3_key)
    remote_size = s3.head_object(Bucket=S3_BUCKET, Key=s3_key)["ContentLength"]
    if remote_size != local_size:
        raise RuntimeError(
            f"UPLOAD VERIFICATION FAILED for {s3_key}\n"
            f"  Local  : {local_size:,} bytes\n"
            f"  S3     : {remote_size:,} bytes"
        )
    print(f"  Verified: {remote_size:,} bytes match.")

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS — RETINAFACE CROP ENGINE
# Identical to validated production engine from previous runs.
# 1080p bypass: downscale 0.5x for detection, upscale box coordinates back.
# ─────────────────────────────────────────────────────────────────────────────

def master_crop_engine(img_path, save_path, bk_tree):
    try:
        img_cv = cv2.imread(img_path)
        if img_cv is None:
            return False, "Corrupted Image", 0.0, 0.0, "", "[]"

        height, width = img_cv.shape[:2]

        # 1080p bypass — downscale for detection only
        scale_factor = 0.5
        small_cv     = cv2.resize(img_cv, (0, 0), fx=scale_factor, fy=scale_factor)

        faces = RetinaFace.detect_faces(small_cv)
        if not isinstance(faces, dict) or len(faces) == 0:
            return False, "No Face Detected", 0.0, 0.0, "", "[]"

        # Select largest face
        largest_area = 0
        best_face    = None
        for face in faces.values():
            box  = face.get("facial_area")
            if box is None:
                continue
            area = (box[2] - box[0]) * (box[3] - box[1])
            if area > largest_area:
                largest_area = area
                best_face    = face

        if best_face is None:
            return False, "No Valid Face Found", 0.0, 0.0, "", "[]"

        # Upscale box back to original resolution
        raw_box = best_face["facial_area"]
        box     = [int(coord / scale_factor) for coord in raw_box]
        str_box = f"[{box[0]}, {box[1]}, {box[2]}, {box[3]}]"

        confidence = best_face["score"]
        if confidence < CONFIDENCE_THRESHOLD:
            return False, "Low Confidence", confidence, 0.0, "", str_box

        face_w = box[2] - box[0]
        face_h = box[3] - box[1]

        if face_w < MIN_FACE_SIZE or face_h < MIN_FACE_SIZE:
            return False, "Resolution Too Small", confidence, 0.0, "", str_box

        if (face_w * face_h) / (width * height) < MIN_FACE_RATIO:
            return False, "Face Ratio Too Small", confidence, 0.0, "", str_box

        x_min = max(0,      box[0] - PADDING)
        y_min = max(0,      box[1] - PADDING)
        x_max = min(width,  box[2] + PADDING)
        y_max = min(height, box[3] + PADDING)

        cropped_cv = img_cv[y_min:y_max, x_min:x_max]
        gray_crop  = cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2GRAY)
        blur_val   = cv2.Laplacian(gray_crop, cv2.CV_64F).var()

        if blur_val < BLUR_THRESHOLD:
            return False, "Motion Blur Detected", confidence, blur_val, "", str_box

        cropped_pil  = Image.fromarray(cv2.cvtColor(cropped_cv, cv2.COLOR_BGR2RGB))
        standardized = cropped_pil.resize(RESIZE_TARGET, Image.BICUBIC)

        new_hash = imagehash.phash(standardized)
        if bk_tree.find(new_hash, PHASH_HAMMING_MAX):
            return False, f"Duplicate (Hamming<={PHASH_HAMMING_MAX})", confidence, blur_val, str(new_hash), str_box

        bk_tree.add(new_hash)
        standardized.save(save_path, format="JPEG", quality=95)

        return True, "Accepted", confidence, blur_val, str(new_hash), str_box

    except Exception as exc:
        return False, f"Engine Error: {exc}", 0.0, 0.0, "", "[]"

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS — MANIFEST
# ─────────────────────────────────────────────────────────────────────────────

def write_manifest(
    accepted, real_quotas, fake_quotas,
    method_totals, rejection_reasons,
    zip_path, start_time,
):
    timestamp  = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    elapsed    = time.time() - start_time
    zip_size   = os.path.getsize(zip_path) / (1024**3) if os.path.exists(zip_path) else 0.0
    total_acc  = sum(accepted.values())
    total_rej  = sum(rejection_reasons.values())
    all_quotas = {**real_quotas, **fake_quotas}

    lines = [
        "=" * 70,
        "  CELL 5 — RETINAFACE FACE EXTRACTION MANIFEST",
        "=" * 70,
        f"  Timestamp              : {timestamp}",
        f"  Elapsed                : {elapsed:.0f}s ({elapsed/60:.1f} min)",
        f"  Input S3 object        : s3://{S3_BUCKET}/{S3_INPUT_KEY}",
        "",
        "  Quality thresholds:",
        f"    Confidence           : >= {CONFIDENCE_THRESHOLD}",
        f"    Min face size        : {MIN_FACE_SIZE} px",
        f"    Min face ratio       : {MIN_FACE_RATIO}",
        f"    Blur threshold       : {BLUR_THRESHOLD} (Laplacian variance)",
        f"    pHash Hamming max    : <= {PHASH_HAMMING_MAX}",
        f"    Padding              : {PADDING} px",
        f"    Resize target        : {RESIZE_TARGET[0]}x{RESIZE_TARGET[1]} (bicubic)",
        "",
        "  Split proportions: train=70%, val=15%, test=15%",
        "  Fake quota distribution: proportional by method video count",
        "",
        "  Real quotas (70/15/15):",
    ]
    for k, target in sorted(real_quotas.items()):
        got  = accepted.get(k, 0)
        flag = "" if got >= target else "  *** UNDERFILLED ***"
        lines.append(f"    {k:<35}: {got:>6,} / {target:>6,}{flag}")

    lines += ["", "  Fake quotas per method (proportional):"]
    for method in DF40_METHODS:
        mtotal = method_totals[method]
        macc   = sum(accepted.get(quota_key_fake(s, method), 0) for s in SPLITS)
        lines.append(f"    {method:<20}: {macc:>6,} / {mtotal:>6,}")
        for split in SPLITS:
            k      = quota_key_fake(split, method)
            got    = accepted.get(k, 0)
            target = fake_quotas[k]
            flag   = "" if got >= target else "  *** UNDERFILLED ***"
            lines.append(f"      {split:<6}: {got:>6,} / {target:>6,}{flag}")

    lines += [
        "",
        "=" * 70,
        f"  Total accepted         : {total_acc:,}",
        f"  Total rejected         : {total_rej:,}",
        "",
        "  Rejection reasons:",
    ]
    for reason, count in sorted(rejection_reasons.items(), key=lambda x: -x[1]):
        lines.append(f"    {reason:<45}: {count:>6,}")

    lines += [
        "",
        f"  Output zip size        : {zip_size:.2f} GB",
        f"  S3 zip                 : s3://{S3_BUCKET}/{S3_OUT_KEYS['zip']}",
        f"  S3 manifest            : s3://{S3_BUCKET}/{S3_OUT_KEYS['manifest']}",
        f"  S3 CSV log             : s3://{S3_BUCKET}/{S3_OUT_KEYS['csv']}",
        "=" * 70,
    ]

    with open(MANIFEST_PATH, "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"  Manifest written: {MANIFEST_PATH}")

# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

start_time    = time.time()
timestamp_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

print("=" * 70)
print("  CELL 5 — RETINAFACE FACE EXTRACTION ENGINE")
print(f"  {timestamp_str}")
print(f"  Real cap               : {REAL_CAP:,}")
print(f"  Confidence threshold   : >= {CONFIDENCE_THRESHOLD}")
print(f"  Blur threshold         : >= {BLUR_THRESHOLD}")
print(f"  pHash Hamming max      : <= {PHASH_HAMMING_MAX}")
print(f"  Resize target          : {RESIZE_TARGET[0]}x{RESIZE_TARGET[1]}")
print("=" * 70)

# ─── [1] SETUP ───────────────────────────────────────────────────────────────

print("\n[1/10] Setting up staging directories...")
for d in [RAW_ZIP_DIR, RAW_FRAMES_DIR, FINAL_FACES_DIR, LOGS_DIR]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)

# Pre-create output split directories
for split in SPLITS:
    os.makedirs(os.path.join(FINAL_FACES_DIR, split, "real"), exist_ok=True)
    os.makedirs(os.path.join(FINAL_FACES_DIR, split, "fake"), exist_ok=True)

print(f"  Stage root: {STAGE_DIR}")

# ─── [2] DOWNLOAD RAW FRAMES ZIP FROM S3 ─────────────────────────────────────

print("\n[2/10] Downloading raw frames zip from S3...")
download_from_s3(S3_INPUT_KEY, LOCAL_ZIP_IN)

# ─── [3] UNPACK ──────────────────────────────────────────────────────────────

print("\n[3/10] Unpacking raw frames (native unzip)...")
try:
    subprocess.run(
        ["unzip", "-q", LOCAL_ZIP_IN, "-d", RAW_FRAMES_DIR],
        check=True
    )
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"unzip failed with return code {e.returncode}")

print(f"  Unpacked -> {RAW_FRAMES_DIR}")
os.remove(LOCAL_ZIP_IN)
print(f"  Input zip removed from local SSD.")

# ─── [4] VALIDATE STRUCTURE ──────────────────────────────────────────────────

print("\n[4/10] Validating unpacked folder structure...")

expected_dirs = []
for split in SPLITS:
    expected_dirs.append(os.path.join(RAW_FRAMES_DIR, split, "real"))
    for method in DF40_METHODS:
        expected_dirs.append(os.path.join(RAW_FRAMES_DIR, split, "fake", method))

missing = [d for d in expected_dirs if not os.path.isdir(d)]
if missing:
    raise RuntimeError(
        "Structure validation FAILED. Missing directories:\n" +
        "\n".join(f"  {d}" for d in missing)
    )
print("  Structure validation PASSED.")

# ─── [5] SCAN FRAME PATHS ────────────────────────────────────────────────────

print("\n[5/10] Scanning frame paths...")

real_frames  = {split: [] for split in SPLITS}
fake_frames  = {method: {split: [] for split in SPLITS} for method in DF40_METHODS}

for split in SPLITS:
    real_dir = os.path.join(RAW_FRAMES_DIR, split, "real")
    paths    = []
    for root, dirs, files in os.walk(real_dir):
        for f in files:
            if f.lower().endswith(".jpg"):
                paths.append(os.path.join(root, f))
    real_frames[split] = paths

for method in DF40_METHODS:
    for split in SPLITS:
        method_dir = os.path.join(RAW_FRAMES_DIR, split, "fake", method)
        paths      = []
        for root, dirs, files in os.walk(method_dir):
            for f in files:
                if f.lower().endswith(".jpg"):
                    paths.append(os.path.join(root, f))
        fake_frames[method][split] = paths

total_real_frames = sum(len(v) for v in real_frames.values())
total_fake_frames = sum(
    len(fake_frames[m][s]) for m in DF40_METHODS for s in SPLITS
)

print(f"\n  Real frames scanned:")
for split in SPLITS:
    print(f"    {split:<6}: {len(real_frames[split]):>8,} frames")
print(f"  Total real frames: {total_real_frames:,}")

print(f"\n  Fake frames scanned:")
for method in DF40_METHODS:
    mtotal = sum(len(fake_frames[method][s]) for s in SPLITS)
    print(f"    {method:<20}: {mtotal:>8,} frames")
print(f"  Total fake frames: {total_fake_frames:,}")

if total_real_frames == 0:
    raise RuntimeError("FATAL: Zero real frames found. Check Cell 4 output.")
if total_fake_frames == 0:
    raise RuntimeError("FATAL: Zero fake frames found. Check Cell 4 output.")

# ─── [6] COMPUTE REAL QUOTAS ─────────────────────────────────────────────────

print("\n[6/10] Computing real quotas...")
real_quotas = compute_real_quotas(REAL_CAP)

print(f"  Real cap: {REAL_CAP:,}")
for split in SPLITS:
    k = quota_key_real(split)
    print(f"    {split:<6}: {real_quotas[k]:,}")

# ─── [7] PROCESS REAL FRAMES ─────────────────────────────────────────────────

print("\n[7/10] Processing real frames through RetinaFace engine...")
print(f"  Thresholds: confidence>={CONFIDENCE_THRESHOLD}, "
      f"blur>={BLUR_THRESHOLD}, pHash<={PHASH_HAMMING_MAX}\n")

# Per split-class BK trees for dedup isolation
def _hash_dist(h1, h2):
    return h1 - h2

real_bk_trees = {split: pybktree.BKTree(_hash_dist) for split in SPLITS}

accepted          = {}
rejection_reasons = defaultdict(int)
csv_rows          = []

total_real_target = sum(real_quotas.values())
real_pbar = tqdm(total=total_real_target, desc="Real faces", unit="face")

for split in SPLITS:
    paths = list(real_frames[split])
    random.shuffle(paths)

    qkey    = quota_key_real(split)
    target  = real_quotas[qkey]
    out_dir = os.path.join(FINAL_FACES_DIR, split, "real")
    tree    = real_bk_trees[split]

    accepted[qkey] = 0

    for img_path in paths:
        if accepted[qkey] >= target:
            break

        fname     = os.path.basename(img_path)
        save_path = os.path.join(out_dir, fname)

        passed, reason, conf, blur, phash_val, bbox = master_crop_engine(
            img_path, save_path, tree
        )

        if passed:
            accepted[qkey] += 1
            real_pbar.update(1)
            csv_rows.append([
                img_path, split, "real", "ffpp_original",
                "Accepted", "None",
                round(conf, 4), round(blur, 2), phash_val, bbox
            ])
        else:
            rejection_reasons[reason] += 1
            csv_rows.append([
                img_path, split, "real", "ffpp_original",
                "Rejected", reason,
                round(conf, 4), round(blur, 2), phash_val, bbox
            ])

real_pbar.close()

actual_real_total = sum(accepted.get(quota_key_real(s), 0) for s in SPLITS)
print(f"\n  Real extraction complete")
print(f"  {'Split':<8} {'Accepted':>10} {'Target':>10}")
print(f"  {'-'*8} {'-'*10} {'-'*10}")
for split in SPLITS:
    k = quota_key_real(split)
    print(f"  {split:<8} {accepted.get(k, 0):>10,} {real_quotas[k]:>10,}")
print(f"\n  Actual real total accepted: {actual_real_total:,} / {REAL_CAP:,}")

# Check real quotas
real_underfilled = [
    k for k in real_quotas if accepted.get(k, 0) < real_quotas[k]
]
if real_underfilled:
    print(f"\n  [WARN] {len(real_underfilled)} real quota buckets underfilled:")
    for k in real_underfilled:
        print(f"    {k}: {accepted.get(k,0):,} / {real_quotas[k]:,}")

# ─── COMPUTE FAKE QUOTAS BASED ON ACTUAL REAL YIELD ──────────────────────────

fake_total  = actual_real_total   # match fake cap to real yield exactly
fake_quotas, method_totals = compute_fake_quotas(fake_total)

print(f"\n  Fake total target (matched to real yield): {fake_total:,}")
print(f"  Proportional quotas per method:")
for method in DF40_METHODS:
    print(f"    {method:<20}: {method_totals[method]:,} total  "
          f"(train={fake_quotas[quota_key_fake('train',method)]:,}  "
          f"val={fake_quotas[quota_key_fake('val',method)]:,}  "
          f"test={fake_quotas[quota_key_fake('test',method)]:,})")

# ─── [8] PROCESS FAKE FRAMES ─────────────────────────────────────────────────

print("\n[8/10] Processing fake frames through RetinaFace engine...")

fake_bk_trees = {split: pybktree.BKTree(_hash_dist) for split in SPLITS}

total_fake_target = sum(fake_quotas.values())
fake_pbar = tqdm(total=total_fake_target, desc="Fake faces", unit="face")

for method in DF40_METHODS:
    method_accepted = 0
    method_target   = method_totals[method]

    for split in SPLITS:
        qkey   = quota_key_fake(split, method)
        target = fake_quotas[qkey]
        accepted[qkey] = 0

        paths   = list(fake_frames[method][split])
        random.shuffle(paths)

        out_dir = os.path.join(FINAL_FACES_DIR, split, "fake")
        tree    = fake_bk_trees[split]

        for img_path in paths:
            if accepted[qkey] >= target:
                break

            # Prefix filename with method to avoid collisions in merged fake dir
            fname     = f"{method}_{os.path.basename(img_path)}"
            save_path = os.path.join(out_dir, fname)

            passed, reason, conf, blur, phash_val, bbox = master_crop_engine(
                img_path, save_path, tree
            )

            if passed:
                accepted[qkey] += 1
                method_accepted += 1
                fake_pbar.update(1)
                csv_rows.append([
                    img_path, split, "fake", method,
                    "Accepted", "None",
                    round(conf, 4), round(blur, 2), phash_val, bbox
                ])
            else:
                rejection_reasons[reason] += 1
                csv_rows.append([
                    img_path, split, "fake", method,
                    "Rejected", reason,
                    round(conf, 4), round(blur, 2), phash_val, bbox
                ])

    tqdm.write(f"  {method:<20}: {method_accepted:,} / {method_target:,} accepted")

fake_pbar.close()

actual_fake_total = sum(
    accepted.get(quota_key_fake(s, m), 0)
    for m in DF40_METHODS for s in SPLITS
)
print(f"\n  Actual fake total accepted: {actual_fake_total:,} / {fake_total:,}")

# ─── QUOTA SUMMARY ───────────────────────────────────────────────────────────

print("\n" + "─" * 70)
print("  EXTRACTION SUMMARY")
print("─" * 70)

total_accepted = sum(accepted.values())
total_rejected = sum(rejection_reasons.values())

print(f"  Total accepted : {total_accepted:,}")
print(f"  Total rejected : {total_rejected:,}")
print(f"  Real accepted  : {actual_real_total:,}")
print(f"  Fake accepted  : {actual_fake_total:,}")
print(f"  Balance        : {'50/50 CLEAN' if actual_real_total == actual_fake_total else f'MISMATCH — real={actual_real_total} fake={actual_fake_total}'}")

# Warn only — No fail hard on underfilled quotas
all_quotas  = {**real_quotas, **fake_quotas}
underfilled = {k: (accepted.get(k,0), all_quotas[k]) for k in all_quotas if accepted.get(k,0) < all_quotas[k]}

if underfilled:
    print(f"\n  [WARN] {len(underfilled)} bucket(s) underfilled — proceeding anyway:")
    for k, (got, target) in sorted(underfilled.items()):
        print(f"    {k}: {got:,} / {target:,}  (short by {target-got:,})")
else:
    print("\n  All quota buckets filled.")

print("\n  Proceeding to archive.")

# ─── [9] WRITE CSV LOG ────────────────────────────────────────────────────────

print("\n[9/10] Writing CSV log...")
os.makedirs(LOGS_DIR, exist_ok=True)
with open(CSV_LOG_PATH, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "input_path", "split", "class", "method",
        "status", "reason", "confidence", "blur_score", "phash", "bounding_box"
    ])
    writer.writerows(csv_rows)
print(f"  Rows written: {len(csv_rows):,}")

# ─── DISK CHECK ──────────────────────────────────────────────────────────────

faces_size = sum(
    os.path.getsize(os.path.join(root, f))
    for root, dirs, files in os.walk(FINAL_FACES_DIR)
    for f in files
)
_, _, free = shutil.disk_usage(BASE_DIR)
print(f"\n  Faces size : {faces_size / (1024**3):.2f} GB")
print(f"  EBS free   : {free / (1024**3):.2f} GB")

if free < faces_size * 1.2:
    raise RuntimeError(
        f"Insufficient disk space for zip.\n"
        f"  Faces  : {faces_size / (1024**3):.2f} GB\n"
        f"  Free   : {free / (1024**3):.2f} GB"
    )

# ─── ZIP ─────────────────────────────────────────────────────────────────────

print("\n  Creating final faces archive...")
write_manifest(
    accepted, real_quotas, fake_quotas,
    method_totals, rejection_reasons,
    FINAL_ZIP_BASE + ".zip", start_time,
)

shutil.make_archive(FINAL_ZIP_BASE, "zip", FINAL_FACES_DIR)
final_zip_path = FINAL_ZIP_BASE + ".zip"

if not os.path.isfile(final_zip_path) or os.path.getsize(final_zip_path) == 0:
    raise RuntimeError(f"FATAL: Final zip missing or empty: {final_zip_path}")

zip_size_gb = os.path.getsize(final_zip_path) / (1024**3)
print(f"  Archive ready: {zip_size_gb:.2f} GB -> {final_zip_path}")

# ─── [10] UPLOAD + VERIFY + CLEANUP ──────────────────────────────────────────

print("\n[10/10] Uploading to S3...")

upload_and_verify(final_zip_path, S3_OUT_KEYS["zip"])
upload_and_verify(MANIFEST_PATH,  S3_OUT_KEYS["manifest"])
upload_and_verify(CSV_LOG_PATH,   S3_OUT_KEYS["csv"])

print("\n  All uploads verified.")
print("  Local artifacts preserved — review results before manual cleanup.")
print(f"  Final faces : {FINAL_FACES_DIR}")
print(f"  Raw frames  : {RAW_FRAMES_DIR}")
print(f"  Logs        : {LOGS_DIR}")

_, _, free = shutil.disk_usage(BASE_DIR)
print(f"  EBS free after cleanup: {free / (1024**3):.1f} GB")

# ─── DONE ─────────────────────────────────────────────────────────────────────

total_elapsed = time.time() - start_time
print(f"\n{'=' * 70}")
print("  CELL 5 COMPLETE")
print(f"{'=' * 70}")
print(f"  Real accepted  : {actual_real_total:,}")
print(f"  Fake accepted  : {actual_fake_total:,}")
print(f"  Total faces    : {total_accepted:,}")
print(f"  Total rejected : {total_rejected:,}")
print(f"  Elapsed        : {total_elapsed:.0f}s ({total_elapsed/60:.1f} min)")
print(f"  S3 output      : s3://{S3_BUCKET}/{S3_OUT_KEYS['zip']}")
print(f"  Manifest       : s3://{S3_BUCKET}/{S3_OUT_KEYS['manifest']}")
print(f"  CSV log        : s3://{S3_BUCKET}/{S3_OUT_KEYS['csv']}")
print(f"{'=' * 70}")

  CELL 5 — RETINAFACE FACE EXTRACTION ENGINE
  2026-05-09 10:38:33 UTC
  Real cap               : 22,000
  Confidence threshold   : >= 0.9
  Blur threshold         : >= 25
  pHash Hamming max      : <= 2
  Resize target          : 260x260

[1/10] Setting up staging directories...
  Stage root: /home/ec2-user/SageMaker/df40_run5/retinaface_stage

[2/10] Downloading raw frames zip from S3...
  Saved: /home/ec2-user/SageMaker/df40_run5/retinaface_stage/raw_zip/df40_run5_raw_frames.zip  (11.42 GB)

[3/10] Unpacking raw frames (native unzip)...
  Unpacked -> /home/ec2-user/SageMaker/df40_run5/retinaface_stage/raw_frames
  Input zip removed from local SSD.

[4/10] Validating unpacked folder structure...
  Structure validation PASSED.

[5/10] Scanning frame paths...

  Real frames scanned:
    train :   21,000 frames
    val   :    4,500 frames
    test  :    4,500 frames
  Total real frames: 30,000

  Fake frames scanned:
    fsgan               :    6,860 frames
    faceswap            :   

Real faces:   0%|          | 0/22000 [00:00<?, ?face/s]

Real faces:   0%|          | 59/22000 [03:36<22:20:30,  3.67s/face]



  Real extraction complete
  Split      Accepted     Target
  -------- ---------- ----------
  train        14,179     15,400
  val           3,180      3,300
  test          2,957      3,300

  Actual real total accepted: 20,316 / 22,000

  [WARN] 3 real quota buckets underfilled:
    train_real: 14,179 / 15,400
    val_real: 3,180 / 3,300
    test_real: 2,957 / 3,300

  Fake total target (matched to real yield): 20,316
  Proportional quotas per method:
    fsgan               : 2,268 total  (train=1,588  val=340  test=340)
    faceswap            : 2,381 total  (train=1,667  val=357  test=357)
    simswap             : 3,270 total  (train=2,290  val=490  test=490)
    inswap              : 2,920 total  (train=2,044  val=438  test=438)
    blendface           : 2,354 total  (train=1,648  val=353  test=353)
    mobileswap          : 2,377 total  (train=1,665  val=356  test=356)
    e4s                 : 2,374 total  (train=1,662  val=356  test=356)
    facedancer          : 2,372 tota

Fake faces:   0%|          | 0/20316 [00:00<?, ?face/s]

Fake faces:  20%|██        | 4089/20320 [1:52:47<7:27:41,  1.65s/face]


  fsgan               : 2,268 / 2,268 accepted
  faceswap            : 2,381 / 2,381 accepted
  simswap             : 3,270 / 3,270 accepted
  inswap              : 2,920 / 2,920 accepted
  blendface           : 2,354 / 2,354 accepted
  mobileswap          : 2,377 / 2,377 accepted
  e4s                 : 2,374 / 2,374 accepted
  facedancer          : 2,372 / 2,372 accepted

  Actual fake total accepted: 20,316 / 20,316

──────────────────────────────────────────────────────────────────────
  EXTRACTION SUMMARY
──────────────────────────────────────────────────────────────────────
  Total accepted : 40,632
  Total rejected : 22,823
  Real accepted  : 20,316
  Fake accepted  : 20,316
  Balance        : 50/50 CLEAN

  [WARN] 3 bucket(s) underfilled — proceeding anyway:
    test_real: 2,957 / 3,300  (short by 343)
    train_real: 14,179 / 15,400  (short by 1,221)
    val_real: 3,180 / 3,300  (short by 120)

  Proceeding to archive.

[9/10] Writing CSV log...
  Rows written: 63,455

  Faces